# **Northbridge University Admissions Decision Support**
## **Data loading, cleaning, exploratory analysis and predictive modelling (Naive Bayes and XGBoost)**

**Role:** Data Analytics Lead, Admissions and Recruitment teams

**Purpose.** Northbridge University is considering a machine learning model that supports admissions officers during initial shortlisting. The model is a **decision support tool, not an automated decision maker**. This notebook loads the two recruiter extracts, combines and cleans them, explores the combined data, trains two classifiers and audits them for bias, fairness and transparency.

## **Design in pseudocode**

```text
PROGRAM AdmissionsDecisionSupport
    INPUT   recruiter_A.csv, recruiter_B.xlsx
    OUTPUT  cleaned combined dataset, EDA outputs, two trained models, fairness audit

    LOAD each file with explicit rules
        treat only empty cells as missing
        keep the text "None" as a valid category, not as a missing value
    COMBINE both tables and derive the recruiter from the student_id prefix
    VALIDATE schema, ID uniqueness, allowed labels, numeric ranges, binary fields

    CLEAN
        standardise labels with a documented mapping
        convert ambiguous codes to missing rather than guessing
        remove duplicate IDs and duplicate records
        correct impossible scores to the limits of their scale
        cap extreme distances using an interquartile range fence
        impute remaining missing values (median, mode or "Not Recorded")
        write every change to a cleaning log and a per-row flag
    INDEX the table on student_id

    EXPLORE
        counts, offer rates, mean, median, distributions, relationships
    TEST the exploration code
        recompute every figure with independent logic and with Excel formulas

    SPLIT data into training, validation and test sets, stratified on the outcome
    SELECT features, excluding protected characteristics
    AUDIT proxy risk between allowed features and protected characteristics

    FOR EACH model IN (Naive Bayes, XGBoost)
        TUNE hyperparameters on the training set with cross-validation and log loss
        FIT the model
        EVALUATE on validation and test sets
    COMPARE models with a majority-class baseline and choose using log loss

    AUDIT fairness of out-of-fold predictions for every protected group
    EXPLAIN individual predictions
    SAVE outputs and REPORT key findings
END PROGRAM
```

In [1]:
# Standard library
import os, re, math, json, warnings, platform, random, statistics
from pathlib import Path
from datetime import datetime

# Data handling
import numpy as np
import pandas as pd

# Statistics
from scipy.stats import chi2_contingency, pearsonr, spearmanr, skew

# Interactive visualisation
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Machine learning
import sklearn
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     RandomizedSearchCV, cross_val_predict, cross_validate)
from sklearn.metrics import (log_loss, brier_score_loss, accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, matthews_corrcoef, confusion_matrix,
                             roc_curve, precision_recall_curve)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from scipy.stats import uniform, randint
import joblib

# Excel output
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

from IPython.display import display, Markdown

warnings.filterwarnings('ignore')

# Reproducibility: every random step uses this seed
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

# Colab shows interactive Plotly charts with the dedicated renderer.
# Outside Colab, Plotly selects its own default renderer.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    pio.renderers.default = 'colab'
except ImportError:
    IN_COLAB = False

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

print('Environment ready')
print(f'  Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}')
print(f'  scikit-learn {sklearn.__version__} | xgboost {xgb.__version__} | plotly {__import__("plotly").__version__}')
print(f'  Running in Google Colab: {IN_COLAB}')

# Compatibility guard: some older XGBoost releases fail inside newer scikit-learn pipelines
try:
    Pipeline([('model', XGBClassifier(n_estimators=2, verbosity=0))]).fit(np.random.rand(20, 2), np.r_[np.zeros(10), np.ones(10)])
    print('  XGBoost and scikit-learn work together correctly')
except Exception as error:
    print('  XGBoost compatibility problem:', error)
    print('  Fix: run  !pip install -U xgboost  in a new cell, then use Runtime > Restart session and run all again')

Environment ready
  Python 3.13.18 | pandas 2.2.3 | numpy 2.1.3
  scikit-learn 1.6.1 | xgboost 3.4.1 | plotly 5.24.1
  Running in Google Colab: True
  XGBoost and scikit-learn work together correctly


In [2]:
PLOT_CONFIG = {
    'displaylogo': False,
    'responsive': True,
    'toImageButtonOptions': {'format': 'png', 'scale': 3, 'filename': 'northbridge_chart'},
}

# Continuous colour maps offered in the dropdowns (all are built-in Plotly names)
CMAPS = ['Viridis', 'Plasma', 'Turbo', 'Cividis', 'Inferno', 'Magma', 'Spectral', 'RdYlGn',
         'RdBu', 'YlGnBu', 'Portland', 'Tealrose', 'Sunsetdark', 'Blues', 'Reds']

# Categorical colour combinations offered in the dropdowns
PALETTES = {
    'Bold': px.colors.qualitative.Bold, 'Vivid': px.colors.qualitative.Vivid,
    'Safe': px.colors.qualitative.Safe, 'Set1': px.colors.qualitative.Set1,
    'Dark24': px.colors.qualitative.Dark24, 'Pastel': px.colors.qualitative.Pastel,
    'Prism': px.colors.qualitative.Prism, 'Antique': px.colors.qualitative.Antique,
}

# Meaningful fixed colours used consistently across the notebook
OUTCOME_COLOURS = {'Offered': '#2A9D8F', 'Not offered': '#E76F51'}
GENDER_COLOURS = {'Female': '#B5179E', 'Male': '#3A86FF', 'Other': '#FFB703'}
RECRUITER_COLOURS = {'Recruiter A': '#264653', 'Recruiter B': '#F4A261'}
MODEL_COLOURS = {'Naive Bayes': '#7209B7', 'XGBoost': '#F77F00', 'Majority baseline': '#8D99AE'}


def show(fig):
    """Display a Plotly figure with the shared interactive configuration."""
    fig.show(config=PLOT_CONFIG)


def style_figure(fig, title, subtitle=None, height=520, top_margin=170):
    """Apply the shared layout: white template, bold title, optional subtitle and room for dropdowns."""
    text = f'<b>{title}</b>'
    if subtitle:
        text += f'<br><span style="font-size:13px;color:#555555">{subtitle}</span>'
    fig.update_layout(
        template='plotly_white',
        title=dict(text=text, x=0.01, y=0.96, yanchor='top'),
        height=height,
        margin=dict(t=top_margin, l=70, r=85, b=70),
        font=dict(family='Arial, sans-serif', size=13),
        hoverlabel=dict(font_size=13),
    )
    return fig


def _menu_position(fig):
    """Return the paper y-coordinate just above the plotting area, used to place dropdowns."""
    plot_height = fig.layout.height - fig.layout.margin.t - fig.layout.margin.b
    return 1 + 42 / plot_height, 1 + 78 / plot_height


def _add_menu(fig, buttons, label, x, active):
    y_menu, y_label = _menu_position(fig)
    menu = dict(type='dropdown', buttons=buttons, active=active, x=x, xanchor='left',
                y=y_menu, yanchor='bottom', showactive=True, bgcolor='#F8F9FA',
                bordercolor='#ADB5BD', font=dict(size=12))
    fig.update_layout(updatemenus=list(fig.layout.updatemenus) + [menu])
    fig.add_annotation(text=f'<b>{label}</b>', x=x, y=y_label, xref='paper', yref='paper',
                       xanchor='left', yanchor='bottom', showarrow=False, font=dict(size=12, color='#495057'))


def add_cmap_dropdown(fig, attr='colorscale', trace_ids=None, initial='Viridis', x=0.0):
    """Add an interactive colour map dropdown.

    attr      : the Plotly attribute that holds the colour scale ('colorscale' for heatmaps,
                'marker.colorscale' for bars and scatter points)
    trace_ids : which traces the dropdown controls (default: all traces)
    initial   : the colour map the figure starts with (it must also be set on the traces)
    """
    ids = list(range(len(fig.data))) if trace_ids is None else list(trace_ids)
    buttons = [dict(label=name, method='restyle', args=[{attr: name}, ids]) for name in CMAPS]
    _add_menu(fig, buttons, 'Colour map', x, CMAPS.index(initial))


def add_palette_dropdown(fig, trace_ids, x=0.0, groups=None, line=False):
    """Add an interactive dropdown that swaps the colour combination of categorical traces.

    The first option restores the original, meaningful colours.
    groups : optional list with one group number per trace, so traces in different subplots that
             represent the same category receive the same colour (default: one colour per trace).
    line   : set to True for line traces, so the line colour changes as well as the marker colour.
    Only traces whose colour is a single colour (not an array) are supported.
    """
    ids = list(trace_ids)
    groups = list(groups) if groups is not None else list(range(len(ids)))
    original = {'marker.color': [fig.data[i].marker.color for i in ids]}
    if line:
        original['line.color'] = [fig.data[i].line.color for i in ids]
    buttons = [dict(label='Original colours', method='restyle', args=[original, ids])]
    for name, colours in PALETTES.items():
        chosen = [colours[g % len(colours)] for g in groups]
        change = {'marker.color': chosen}
        if line:
            change['line.color'] = chosen
        buttons.append(dict(label=name, method='restyle', args=[change, ids]))
    _add_menu(fig, buttons, 'Colour combination', x, 0)


def value_bar(labels, values, title, subtitle=None, axis_title='Count', text=None,
              horizontal=False, height=520, value_format=',.0f', hover_unit='', colour_range=None, text_inside=False):
    """Labelled bar chart coloured by value, with an interactive colour map dropdown."""
    labels, values = list(labels), list(values)
    text = text if text is not None else [format(v, value_format) for v in values]
    marker = dict(color=values, colorscale='Viridis', showscale=True,
                  colorbar=dict(title=axis_title), line=dict(color='white', width=1))
    if colour_range:
        marker.update(cmin=colour_range[0], cmax=colour_range[1])
    position = 'inside' if text_inside else 'outside'
    if horizontal:
        trace = go.Bar(y=labels, x=values, orientation='h', text=text, textposition=position, insidetextanchor='start',
                       marker=marker, cliponaxis=False,
                       hovertemplate='%{y}<br>%{x:,.2f}' + hover_unit + '<extra></extra>')
    else:
        trace = go.Bar(x=labels, y=values, text=text, textposition=position,
                       marker=marker, cliponaxis=False,
                       hovertemplate='%{x}<br>%{y:,.2f}' + hover_unit + '<extra></extra>')
    fig = go.Figure(trace)
    style_figure(fig, title, subtitle, height)
    if horizontal:
        fig.update_xaxes(title=axis_title)
        fig.update_yaxes(autorange='reversed')
    else:
        fig.update_yaxes(title=axis_title)
    add_cmap_dropdown(fig, 'marker.colorscale')
    return fig


def plotly_table(df, title, status_col=None, height=None, decimals=3, col_widths=None):
    """Colour-coded interactive table. A status column is shaded green, amber or red."""
    d = df.copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            whole = d[c].dropna().apply(lambda v: float(v).is_integer()).all() if d[c].notna().any() else False
            d[c] = d[c].map(lambda v: '' if pd.isna(v) else (f'{v:,.0f}' if whole else f'{v:,.{decimals}f}'))
    n = len(d)
    stripe = ['#F1FAEE' if i % 2 == 0 else '#E3EEF2' for i in range(n)]
    fills = [stripe[:] for _ in d.columns]
    if status_col and status_col in d.columns:
        palette = {'pass': '#B7E4C7', 'match': '#B7E4C7', 'yes': '#B7E4C7', 'best': '#B7E4C7',
                   'attention': '#FFE8A3', 'review': '#FFE8A3', 'flag': '#FFE8A3', 'caution': '#FFE8A3',
                   'fail': '#F4A6A6', 'check': '#F4A6A6', 'no': '#EDEDED'}
        position = list(d.columns).index(status_col)
        fills[position] = [palette.get(str(v).lower().split()[0], stripe[i]) if str(v) else stripe[i]
                           for i, v in enumerate(d[status_col])]
    fig = go.Figure(go.Table(
        columnwidth=col_widths,
        header=dict(values=[f'<b>{c}</b>' for c in d.columns], fill_color='#264653',
                    font=dict(color='white', size=13), align='left', height=34),
        cells=dict(values=[d[c].astype(str).tolist() for c in d.columns], fill_color=fills,
                   font=dict(color='#1D3557', size=12), align='left', height=28)))
    fig.update_layout(template='plotly_white', title=dict(text=f'<b>{title}</b>', x=0.01),
                      height=height or min(110 + 30 * n, 760), margin=dict(t=70, l=20, r=20, b=20),
                      font=dict(family='Arial, sans-serif'))
    return fig


print('Plotting helpers ready')

Plotting helpers ready


## Load the recruiter files

**How the data is loaded**
- `recruiter_A.csv` is read with `pandas.read_csv` and `recruiter_B.xlsx` with `pandas.read_excel` (engine `openpyxl`)
- Both files use **identical, explicit parsing rules** so that the same value is never read two different ways
- Text columns are read as text and numeric columns are inferred as numbers

In [3]:
CSV_NAME = 'recruiter_A.csv'
XLSX_NAME = 'recruiter_B.xlsx'
SEARCH_DIRS = [Path('.'), Path('/content'), Path('/content/drive/MyDrive'),
               Path('/mnt/user-data/uploads'), Path('data')]

EXPECTED_COLUMNS = ['student_id', 'prior_attainment_score', 'admissions_test_score',
                    'personal_statement_score', 'school_type', 'intended_subject_area',
                    'distance_from_campus_km', 'first_generation_student', 'gender',
                    'ethnicity', 'disability_status', 'admission_outcome']
TEXT_COLUMNS = ['student_id', 'school_type', 'intended_subject_area', 'gender',
                'ethnicity', 'disability_status']
CATEGORICAL_COLUMNS = ['school_type', 'intended_subject_area', 'gender', 'ethnicity', 'disability_status']
SCORE_COLUMNS = ['prior_attainment_score', 'admissions_test_score', 'personal_statement_score']
NUMERIC_COLUMNS = SCORE_COLUMNS + ['distance_from_campus_km', 'first_generation_student', 'admission_outcome']
TARGET = 'admission_outcome'


def locate_file(filename):
    """Find an input file in the usual places, or ask for an upload when running in Colab."""
    for folder in SEARCH_DIRS:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    if IN_COLAB:
        from google.colab import files
        print(f'{filename} was not found. Please select it in the upload dialog.')
        uploaded = files.upload()
        stem = Path(filename).stem.lower()
        for name in uploaded:
            if name == filename or name.lower().startswith(stem):
                return Path(name)
    raise FileNotFoundError(f'Could not find {filename}. Upload it to the Colab files panel and run again.')


class RecruiterDataLoader:
    """Load the recruiter extracts with identical, explicit parsing rules.

    Only empty cells are treated as missing. The text 'None' is preserved because it is a
    valid category in disability_status.
    """

    def __init__(self, csv_path, xlsx_path, sheet_name=0):
        self.csv_path = Path(csv_path)
        self.xlsx_path = Path(xlsx_path)
        self.sheet_name = sheet_name

    @staticmethod
    def _parsing_rules():
        return dict(keep_default_na=False, na_values=[''])

    def load_csv(self):
        frame = pd.read_csv(self.csv_path, dtype={c: str for c in TEXT_COLUMNS},
                            skipinitialspace=True, **self._parsing_rules())
        frame['source_file'] = self.csv_path.name
        return frame

    def load_excel(self):
        frame = pd.read_excel(self.xlsx_path, sheet_name=self.sheet_name, engine='openpyxl',
                              dtype={c: str for c in TEXT_COLUMNS}, **self._parsing_rules())
        frame['source_file'] = self.xlsx_path.name
        return frame


csv_path, xlsx_path = locate_file(CSV_NAME), locate_file(XLSX_NAME)
loader = RecruiterDataLoader(csv_path, xlsx_path)
recruiter_a = loader.load_csv()
recruiter_b = loader.load_excel()

# Demonstration of the default parsing trap described above
default_parse = pd.read_csv(csv_path)
print('Default pandas parsing reads', f"{default_parse['disability_status'].isna().sum():,}",
      "cells of disability_status as missing, although the applicants answered 'None'.")
print('Explicit parsing keeps', f"{(recruiter_a['disability_status'] == 'None').sum():,}",
      "answers as the category 'None' and finds", f"{recruiter_a['disability_status'].isna().sum():,}",
      'genuinely missing cells.')

print(f'\nrecruiter_A.csv : {recruiter_a.shape[0]:,} rows x {recruiter_a.shape[1]} columns')
print(f'recruiter_B.xlsx: {recruiter_b.shape[0]:,} rows x {recruiter_b.shape[1]} columns')
display(recruiter_a.head(5))
display(recruiter_b.head(5))

Default pandas parsing reads 3,094 cells of disability_status as missing, although the applicants answered 'None'.
Explicit parsing keeps 3,094 answers as the category 'None' and finds 0 genuinely missing cells.

recruiter_A.csv : 4,000 rows x 13 columns
recruiter_B.xlsx: 4,000 rows x 13 columns


,student_id,prior_attainment_score,admissions_test_score,personal_statement_score,school_type,intended_subject_area,distance_from_campus_km,first_generation_student,gender,ethnicity,disability_status,admission_outcome,source_file
0,A00001,59.1,59.3,2.94,State,Humanities,47.3,0,Female,Other,None,1,recruiter_A.csv
1,A00002,80.0,78.0,3.60,State,Humanities,49.9,1,Male,Mixed,None,1,recruiter_A.csv
2,A00003,72.8,60.4,3.31,State,Arts,145.9,0,Male,Asian,None,1,recruiter_A.csv
3,A00004,54.9,60.7,3.52,State,STEM,63.1,0,Female,White,None,1,recruiter_A.csv
4,A00005,64.2,56.8,2.99,State,Arts,175.9,1,Male,Black,None,1,recruiter_A.csv


,student_id,prior_attainment_score,admissions_test_score,personal_statement_score,school_type,intended_subject_area,distance_from_campus_km,first_generation_student,gender,ethnicity,disability_status,admission_outcome,source_file
0,B00001,61.3,76.1,3.80,State,Arts,95.6,0,Female,Other,None,0,recruiter_B.xlsx
1,B00002,56.6,60.9,2.81,State,Humanities,80.9,0,Female,Other,None,1,recruiter_B.xlsx
2,B00003,72.4,73.5,3.56,Independent,STEM,70.5,1,Male,White,None,1,recruiter_B.xlsx
3,B00004,76.3,75.4,2.46,State,Social Sciences,118.4,0,Male,Other,None,1,recruiter_B.xlsx
4,B00005,65.1,57.8,3.75,State,Humanities,88.3,1,Female,White,Learning,1,recruiter_B.xlsx


## Combine the two recruiter datasets

The two tables share the same twelve columns, so they are stacked with `pandas.concat`. The recruiter is derived from the **prefix of `student_id`** (A or B) and cross-checked against the source file. The result is kept as the **raw combined dataset**, so the before and after of cleaning can always be compared.

In [4]:
column_check = {
    'Recruiter A has all expected columns': set(EXPECTED_COLUMNS).issubset(recruiter_a.columns),
    'Recruiter B has all expected columns': set(EXPECTED_COLUMNS).issubset(recruiter_b.columns),
    'Both files share identical column names': set(recruiter_a.columns) == set(recruiter_b.columns),
}
for message, passed in column_check.items():
    print(('Pass  ' if passed else 'FAIL  ') + message)

combined_raw = pd.concat([recruiter_a, recruiter_b], ignore_index=True)
combined_raw['recruiter'] = 'Recruiter ' + combined_raw['student_id'].str.extract(r'^([A-Za-z]+)')[0].str.upper()

print(f'\nCombined raw dataset: {combined_raw.shape[0]:,} rows x {combined_raw.shape[1]} columns')
display(combined_raw['recruiter'].value_counts().rename('applications').to_frame())

Pass  Recruiter A has all expected columns
Pass  Recruiter B has all expected columns
Pass  Both files share identical column names

Combined raw dataset: 8,000 rows x 14 columns


,applications
recruiter,
Recruiter A,4000
Recruiter B,4000


## Validate the data

Before any cleaning, the combined data is checked against the data dictionary in the brief:
- every expected column exists and student IDs are unique and well formed
- the ID prefix agrees with the source file
- categorical columns only use the accepted labels
- scores stay inside their scale (0 to 100 for attainment and admissions test, 1 to 5 for the personal statement)
- binary fields contain only 0 or 1
- no cells are missing

In [5]:
ALLOWED_VALUES = {
    'school_type': ['State', 'Independent', 'International'],
    'intended_subject_area': ['STEM', 'Humanities', 'Social Sciences', 'Arts'],
    'gender': ['Male', 'Female', 'Other'],
    'ethnicity': ['White', 'Black', 'Asian', 'Mixed', 'Other'],
    'disability_status': ['None', 'Physical', 'Learning', 'Mental Health'],
}
SCORE_LIMITS = {'prior_attainment_score': (0, 100), 'admissions_test_score': (0, 100),
                'personal_statement_score': (1, 5)}


def validate_dataset(df, extra_allowed=None):
    """Run the validation checks and return a tidy table of results."""
    d = df.reset_index() if df.index.name == 'student_id' else df.copy()
    extra_allowed = extra_allowed or {}
    results = []

    def add(check, failed, detail=''):
        results.append({'Check': check, 'Failed rows': int(failed),
                        'Status': 'Pass' if failed == 0 else 'Attention', 'Detail': detail})

    missing_columns = sorted(set(EXPECTED_COLUMNS) - set(d.columns))
    add('All expected columns are present', len(missing_columns), ', '.join(missing_columns))
    add('student_id values are unique', d['student_id'].duplicated().sum())
    add('student_id has one letter followed by five digits',
        (~d['student_id'].astype(str).str.match(r'^[A-Za-z]\d{5}$')).sum())
    add('student_id prefix agrees with the recruiter',
        (d['student_id'].str[0].str.upper() != d['recruiter'].str[-1]).sum())

    for column, allowed in ALLOWED_VALUES.items():
        permitted = allowed + extra_allowed.get(column, [])
        bad = d[column].notna() & ~d[column].isin(permitted)
        detail = ', '.join(sorted(d.loc[bad, column].astype(str).unique())[:8])
        add(f'{column} uses only accepted labels', bad.sum(), detail)

    for column, (low, high) in SCORE_LIMITS.items():
        bad = d[column].notna() & ((d[column] < low) | (d[column] > high))
        add(f'{column} lies between {low} and {high}', bad.sum())
    add('distance_from_campus_km is not negative', (d['distance_from_campus_km'] < 0).sum())
    for column in ['first_generation_student', TARGET]:
        add(f'{column} contains only 0 or 1', (d[column].notna() & ~d[column].isin([0, 1])).sum())

    add('No missing cells in the twelve data columns', d[EXPECTED_COLUMNS].isna().sum().sum())
    return pd.DataFrame(results)


validation_before = validate_dataset(combined_raw)
show(plotly_table(validation_before, 'Validation of the raw combined data', status_col='Status',
                  col_widths=[420, 90, 90, 260], height=720))

In [6]:
# Profile the quality issues by recruiter so the source of each problem is visible
def profile_quality(raw):
    """Count the main quality issues per recruiter (before cleaning)."""
    rows = []
    distance = pd.to_numeric(raw['distance_from_campus_km'], errors='coerce')
    q1, q3 = distance.quantile(0.25), distance.quantile(0.75)
    fence = q3 + 3.0 * (q3 - q1)
    for recruiter, part in raw.groupby('recruiter'):
        labels = sum((part[c].notna() & ~part[c].isin(allowed)).sum() for c, allowed in ALLOWED_VALUES.items())
        missing = int(part[EXPECTED_COLUMNS].isna().sum().sum())
        scores = sum(((part[c] < lo) | (part[c] > hi)).sum() for c, (lo, hi) in SCORE_LIMITS.items())
        extreme = int((part['distance_from_campus_km'] > fence).sum())
        rows += [{'Recruiter': recruiter, 'Issue': 'Non-standard or abbreviated labels', 'Cells': int(labels)},
                 {'Recruiter': recruiter, 'Issue': 'Missing cells', 'Cells': missing},
                 {'Recruiter': recruiter, 'Issue': 'Scores outside valid scale', 'Cells': int(scores)},
                 {'Recruiter': recruiter, 'Issue': 'Extreme distances', 'Cells': extreme}]
    return pd.DataFrame(rows)


quality = profile_quality(combined_raw)
fig = go.Figure()
for recruiter, colour in RECRUITER_COLOURS.items():
    part = quality[quality['Recruiter'] == recruiter]
    fig.add_trace(go.Bar(x=part['Issue'], y=part['Cells'], name=recruiter, marker_color=colour,
                         text=part['Cells'], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}<br>%{y:,} cells<extra>' + recruiter + '</extra>'))
fig.update_layout(barmode='group', legend=dict(orientation='h', y=-0.2))
style_figure(fig, 'Data quality issues found before cleaning, by recruiter',
             'Number of cells affected by each type of issue in each recruiter file')
fig.update_yaxes(title='Cells affected')
add_palette_dropdown(fig, trace_ids=[0, 1])
show(fig)

## Clean the admission data

All cleaning is done by the `AdmissionsCleaner` class so that the same rules can be reused and maintained. **Every change is written to a cleaning log**, and every affected row receives a `cleaning_flags` entry, so nothing is changed silently.

| Issue | Treatment | Reason |
|---|---|---|
| Whitespace and inconsistent case | Trimmed and mapped to the accepted label | Prevents the same category appearing under several names |
| Abbreviated labels such as `S`, `Soc`, `A`, `M`, `F`, `Mental` | Mapped to the full accepted label | Each abbreviation has only one possible meaning |
| Ambiguous code `I` in `school_type` | Set to missing, then imputed | It could mean Independent or International, and guessing would add bias |
| Duplicate IDs or duplicate records | Removed (first record kept) | Duplicates would double-count applicants |
| Scores outside their scale | Capped at the scale limits | Values such as 105.7 on a 0 to 100 scale are impossible |
| Extreme distances beyond three interquartile ranges | Capped at that fence | Keeps genuine long distances while limiting distortion |
| Missing numeric values | Median of the combined data | Robust to skew and outliers |
| Missing categories | Most frequent category | Only a handful of cells are affected |
| Missing disability status | Category `Not Recorded` | Avoids asserting that an applicant has no disability when this is unknown |

The medians and modes affect only a handful of cells, so the leakage from computing them before the train and test split is negligible. The model pipeline itself learns scaling and encoding on the training data only.

In [7]:
class AdmissionsCleaner:
    """Apply documented cleaning rules to the combined recruiter data and keep an audit trail."""

    LABEL_ABBREVIATIONS = {
        'school_type': {'s': 'State'},
        'intended_subject_area': {'a': 'Arts', 'soc': 'Social Sciences'},
        'gender': {'m': 'Male', 'f': 'Female'},
        'disability_status': {'mental': 'Mental Health'},
    }
    # Codes that could mean more than one label are never guessed
    AMBIGUOUS_CODES = {'school_type': ['i']}

    def __init__(self, extreme_iqr_multiplier=3.0):
        self.extreme_iqr_multiplier = extreme_iqr_multiplier
        self.log = []
        self.fill_values = {}

    def _record(self, step, column, issue, rows, action, rationale):
        if rows:
            self.log.append({'Step': step, 'Column': column, 'Issue found': issue,
                             'Rows affected': int(rows), 'Action taken': action, 'Rationale': rationale})

    @staticmethod
    def _flag(flags, mask, text):
        updated = flags.copy()
        updated.loc[mask] = updated.loc[mask].map(lambda s: f'{s};{text}' if s else text)
        return updated

    def _lookup(self, column):
        canonical = {label.lower(): label for label in ALLOWED_VALUES[column]}
        return {**canonical, **self.LABEL_ABBREVIATIONS.get(column, {})}

    def clean(self, raw):
        self.log = []
        df = raw.copy()
        flags = pd.Series('', index=df.index, dtype=object)

        # Text hygiene: trim leading and trailing spaces
        for column in TEXT_COLUMNS:
            before = df[column]
            after = before.str.strip()
            changed = before.notna() & (before != after)
            self._record('Text hygiene', column, 'Leading or trailing spaces', changed.sum(), 'Trimmed',
                         'Spaces create false categories')
            df[column] = after

        # Numeric coercion: anything that is not a number becomes missing and is treated later
        for column in NUMERIC_COLUMNS:
            before = df[column]
            after = pd.to_numeric(before, errors='coerce')
            lost = before.notna() & after.isna()
            self._record('Numeric conversion', column, 'Non-numeric text in a numeric column', lost.sum(),
                         'Converted to missing', 'Cannot be interpreted as a number')
            df[column] = after.astype(float)

        # Label standardisation
        for column in CATEGORICAL_COLUMNS:
            original = df[column]
            key = original.str.lower()
            standard = key.map(self._lookup(column))
            ambiguous = key.isin(self.AMBIGUOUS_CODES.get(column, []))
            unrecognised = original.notna() & standard.isna() & ~ambiguous
            relabelled = original.notna() & standard.notna() & (original != standard)
            examples = ', '.join(sorted(original[relabelled].astype(str).unique())[:6])
            self._record('Label standardisation', column, f'Non-standard labels ({examples})', relabelled.sum(),
                         'Mapped to the accepted label', 'Each variant has a single possible meaning')
            self._record('Label standardisation', column, 'Ambiguous code', ambiguous.sum(),
                         'Set to missing (imputed later)', 'The code could mean more than one category')
            self._record('Label standardisation', column, 'Unrecognised label', unrecognised.sum(),
                         'Set to missing (imputed later)', 'Not in the data dictionary')
            flags = self._flag(flags, relabelled, f'relabelled:{column}')
            flags = self._flag(flags, ambiguous | unrecognised, f'label_to_missing:{column}')
            df[column] = standard

        # Duplicates: repeated IDs, then repeated records under different IDs
        duplicate_ids = df['student_id'].duplicated(keep='first')
        self._record('Duplicates', 'student_id', 'Repeated student ID', duplicate_ids.sum(),
                     'Removed, first record kept', 'An applicant must appear once')
        df, flags = df.loc[~duplicate_ids], flags.loc[~duplicate_ids]
        content_columns = [c for c in EXPECTED_COLUMNS if c != 'student_id']
        duplicate_records = df.duplicated(subset=content_columns, keep='first')
        self._record('Duplicates', 'all columns', 'Identical record under a different ID', duplicate_records.sum(),
                     'Removed, first record kept', 'Likely the same applicant submitted twice')
        df, flags = df.loc[~duplicate_records], flags.loc[~duplicate_records]

        # Target and binary fields must be 0 or 1
        invalid_target = df[TARGET].isna() | ~df[TARGET].isin([0, 1])
        self._record('Target validation', TARGET, 'Missing or invalid outcome', invalid_target.sum(),
                     'Row removed', 'A model cannot learn from or be judged on an unknown outcome')
        df, flags = df.loc[~invalid_target], flags.loc[~invalid_target]
        invalid_binary = df['first_generation_student'].notna() & ~df['first_generation_student'].isin([0, 1])
        self._record('Binary validation', 'first_generation_student', 'Value other than 0 or 1', invalid_binary.sum(),
                     'Set to missing (imputed later)', 'Outside the accepted values')
        if invalid_binary.any():
            df.loc[invalid_binary, 'first_generation_student'] = np.nan

        # Impossible values: scores outside their scale
        for column, (low, high) in SCORE_LIMITS.items():
            below, above = df[column] < low, df[column] > high
            self._record('Range correction', column, f'Below the scale minimum of {low}', below.sum(),
                         f'Raised to {low}', 'Impossible value on the documented scale')
            self._record('Range correction', column, f'Above the scale maximum of {high}', above.sum(),
                         f'Lowered to {high}', 'Impossible value on the documented scale')
            flags = self._flag(flags, below | above, f'score_capped:{column}')
            df[column] = df[column].clip(low, high)

        # Distance: negative values are impossible, extreme values are capped at an IQR fence
        negative = df['distance_from_campus_km'] < 0
        self._record('Range correction', 'distance_from_campus_km', 'Negative distance', negative.sum(),
                     'Set to missing (imputed later)', 'A distance cannot be negative')
        df.loc[negative, 'distance_from_campus_km'] = np.nan
        q1, q3 = df['distance_from_campus_km'].quantile([0.25, 0.75])
        self.distance_fence = q3 + self.extreme_iqr_multiplier * (q3 - q1)
        extreme = df['distance_from_campus_km'] > self.distance_fence
        self._record('Outlier treatment', 'distance_from_campus_km',
                     f'Extreme distance above {self.distance_fence:,.1f} km ({self.extreme_iqr_multiplier} IQR fence)',
                     extreme.sum(), f'Capped at {self.distance_fence:,.1f} km',
                     'Keeps long but plausible distances while limiting distortion')
        flags = self._flag(flags, extreme, 'distance_capped')
        df['distance_from_campus_km'] = df['distance_from_campus_km'].clip(upper=self.distance_fence)

        # Imputation of any remaining missing values
        numeric_to_impute = SCORE_COLUMNS + ['distance_from_campus_km']
        for column in numeric_to_impute:
            missing = df[column].isna()
            if missing.any():
                value = float(df[column].median())
                self.fill_values[column] = value
                self._record('Imputation', column, 'Missing value', missing.sum(),
                             f'Filled with the median ({value:,.2f})', 'Median is robust to skew')
                flags = self._flag(flags, missing, f'imputed:{column}')
                df.loc[missing, column] = value
        for column in ['school_type', 'intended_subject_area', 'gender', 'ethnicity']:
            missing = df[column].isna()
            if missing.any():
                value = df[column].mode().iloc[0]
                self.fill_values[column] = value
                self._record('Imputation', column, 'Missing or ambiguous label', missing.sum(),
                             f"Filled with the most frequent category ('{value}')", 'Only a handful of cells affected')
                flags = self._flag(flags, missing, f'imputed:{column}')
                df.loc[missing, column] = value
        missing = df['disability_status'].isna()
        self._record('Imputation', 'disability_status', 'Missing value', missing.sum(),
                     "Filled with the category 'Not Recorded'",
                     'Avoids claiming that an applicant has no disability when this is unknown')
        flags = self._flag(flags, missing, 'imputed:disability_status')
        df.loc[missing, 'disability_status'] = 'Not Recorded'
        missing = df['first_generation_student'].isna()
        if missing.any():
            value = int(df['first_generation_student'].mode().iloc[0])
            self._record('Imputation', 'first_generation_student', 'Missing value', missing.sum(),
                         f'Filled with the most frequent value ({value})', 'Only a handful of cells affected')
            flags = self._flag(flags, missing, 'imputed:first_generation_student')
            df.loc[missing, 'first_generation_student'] = value

        # Final types, flags and index
        df['first_generation_student'] = df['first_generation_student'].astype(int)
        df[TARGET] = df[TARGET].astype(int)
        df['cleaning_flags'] = flags.loc[df.index]
        ordered = ['student_id', 'recruiter'] + [c for c in EXPECTED_COLUMNS if c != 'student_id'] + ['cleaning_flags']
        df = df[ordered].set_index('student_id', verify_integrity=True).sort_index()
        return df

    def log_frame(self):
        return pd.DataFrame(self.log, columns=['Step', 'Column', 'Issue found', 'Rows affected',
                                               'Action taken', 'Rationale'])


cleaner = AdmissionsCleaner(extreme_iqr_multiplier=3.0)
df = cleaner.clean(combined_raw)
cleaning_log = cleaner.log_frame()

print(f'Rows before cleaning: {len(combined_raw):,} | rows after cleaning: {len(df):,} | '
      f'rows with at least one change: {(df["cleaning_flags"] != "").sum():,}')
show(plotly_table(cleaning_log, 'Cleaning log: every change made to the data', height=760,
                  col_widths=[140, 210, 330, 90, 270, 340]))

Rows before cleaning: 8,000 | rows after cleaning: 8,000 | rows with at least one change: 657


In [8]:
# Confirm the cleaned data now passes every validation check
validation_after = validate_dataset(df, extra_allowed={'disability_status': ['Not Recorded']})
show(plotly_table(validation_after, 'Validation of the cleaned data', status_col='Status',
                  col_widths=[420, 90, 90, 260], height=720))
assert (validation_after['Status'] == 'Pass').all(), 'Cleaned data still fails a validation check'
assert df.index.is_unique and df.index.name == 'student_id', 'Data must be indexed on student_id'
print('The cleaned data passes every validation check and is indexed on student_id.')
display(df.head(8))

The cleaned data passes every validation check and is indexed on student_id.


,recruiter,prior_attainment_score,admissions_test_score,personal_statement_score,school_type,intended_subject_area,distance_from_campus_km,first_generation_student,gender,ethnicity,disability_status,admission_outcome,cleaning_flags
student_id,,,,,,,,,,,,,
A00001,Recruiter A,59.1,59.3,2.94,State,Humanities,47.3,0,Female,Other,None,1,
A00002,Recruiter A,80.0,78.0,3.60,State,Humanities,49.9,1,Male,Mixed,None,1,
A00003,Recruiter A,72.8,60.4,3.31,State,Arts,145.9,0,Male,Asian,None,1,
A00004,Recruiter A,54.9,60.7,3.52,State,STEM,63.1,0,Female,White,None,1,
A00005,Recruiter A,64.2,56.8,2.99,State,Arts,175.9,1,Male,Black,None,1,
A00006,Recruiter A,86.5,66.4,3.99,Independent,STEM,107.2,1,Male,Mixed,Learning,1,
A00007,Recruiter A,45.7,64.8,4.59,State,Social Sciences,197.1,0,Female,Other,None,0,
A00008,Recruiter A,65.7,67.0,1.95,State,STEM,36.8,0,Male,Mixed,None,0,


In [9]:
# Compare the numeric columns before and after cleaning to show the effect on outliers
raw_numeric = combined_raw[['prior_attainment_score', 'admissions_test_score',
                            'personal_statement_score', 'distance_from_campus_km']].apply(pd.to_numeric, errors='coerce')
titles = ['Prior attainment score', 'Admissions test score', 'Personal statement score', 'Distance from campus (km)']
fig = make_subplots(rows=1, cols=4, subplot_titles=titles, horizontal_spacing=0.06)
for position, column in enumerate(raw_numeric.columns, start=1):
    fig.add_trace(go.Box(y=raw_numeric[column], name='Raw', marker_color='#E76F51', boxmean=True,
                         boxpoints='outliers', showlegend=(position == 1), legendgroup='raw'), row=1, col=position)
    fig.add_trace(go.Box(y=df[column], name='Cleaned', marker_color='#2A9D8F', boxmean=True,
                         boxpoints='outliers', showlegend=(position == 1), legendgroup='clean'), row=1, col=position)
style_figure(fig, 'Effect of cleaning on the numeric columns',
             'Box shows the middle half of values, the dashed line is the mean and dots are outliers', height=560)
fig.update_layout(legend=dict(orientation='h', y=-0.12))
add_palette_dropdown(fig, trace_ids=list(range(len(fig.data))))
show(fig)

## Descriptive analysis (exploratory data analysis)

The analysis covers every item required by the brief:
- number of records and percentage of students offered a place
- comparison of places offered by gender
- number of applications and overall offer rate by recruiter
- mean and median of prior attainment, admissions test score and distance from campus
- distributions of those three variables
- the relationship between prior attainment and the admissions test score

Optional items follow: disability distribution, gender proportion, places offered by ethnicity and subject area by gender. **Age distribution cannot be produced because the supplied data dictionary contains no age column.** The notebook checks for one and says so rather than inventing data.

All headline figures are stored in a dictionary called `eda_results`. The next stage tests these stored figures against an independent calculation and against Excel formulas.

In [10]:
eda_results = {}
OUTCOME_NAMES = {1: 'Offered', 0: 'Not offered'}


def chi_square_vs_outcome(data, column):
    """Chi-square test of independence between a column and the admission outcome."""
    table = pd.crosstab(data[column], data[TARGET])
    chi2, p_value, dof, _ = chi2_contingency(table)
    k = min(table.shape) - 1
    cramers_v = math.sqrt(chi2 / (table.values.sum() * k)) if k > 0 else np.nan
    return {'Variable': column, 'Chi-square': chi2, 'Degrees of freedom': dof, 'p-value': p_value,
            "Cramer's V": cramers_v, 'Significant at 5 percent': 'Flag' if p_value < 0.05 else 'No'}


def outcome_by_group(column, order=None, title='', subtitle='', minimum_group=10):
    """Table and chart of applications, offers and offer rate for each group of a column.

    Groups with fewer than minimum_group applicants stay in the table but are left out of the chart,
    because a rate based on a handful of people is misleading.
    """
    table = df.groupby(column)[TARGET].agg(applications='size', offers='sum')
    if order:
        table = table.reindex(order)
    table['not_offered'] = table['applications'] - table['offers']
    table['offer_rate_pct'] = table['offers'] / table['applications'] * 100
    shown = table[table['applications'] >= minimum_group]
    overall = df[TARGET].mean() * 100
    fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.1,
                        subplot_titles=('Applicants offered and not offered', f'Offer rate (%); dashed line is the overall {overall:.1f}%'))
    fig.add_trace(go.Bar(x=shown.index, y=shown['offers'], name='Offered', marker_color=OUTCOME_COLOURS['Offered'],
                         text=shown['offers'], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}<br>Offered: %{y:,}<extra></extra>'), row=1, col=1)
    fig.add_trace(go.Bar(x=shown.index, y=shown['not_offered'], name='Not offered', marker_color=OUTCOME_COLOURS['Not offered'],
                         text=shown['not_offered'], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}<br>Not offered: %{y:,}<extra></extra>'), row=1, col=1)
    fig.add_trace(go.Bar(x=shown.index, y=shown['offer_rate_pct'], name='Offer rate (%)', marker_color='#1D3557',
                         text=[f'{v:.1f}%' for v in shown['offer_rate_pct']], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}<br>Offer rate: %{y:.2f}%<extra></extra>'), row=1, col=2)
    fig.add_hline(y=overall, line_dash='dash', line_color='#E63946', line_width=2, row=1, col=2)
    fig.update_layout(barmode='group', legend=dict(orientation='h', y=-0.18))
    style_figure(fig, title, subtitle)
    fig.update_yaxes(title_text='Applicants', range=[0, shown[['offers', 'not_offered']].values.max() * 1.15], row=1, col=1)
    fig.update_yaxes(title_text='Offer rate (%)', range=[0, 100], row=1, col=2)
    add_palette_dropdown(fig, trace_ids=[0, 1, 2], groups=[0, 1, 2])
    return table, fig


# Number of records and percentage offered a place
eda_results['Number of records'] = len(df)
eda_results['Number of offers'] = int(df[TARGET].sum())
eda_results['Percentage offered a place'] = df[TARGET].mean()

fig = go.Figure()
cards = [('Applications', len(df), '#264653', ',.0f', ''), ('Offers made', eda_results['Number of offers'], '#2A9D8F', ',.0f', ''),
         ('Not offered', len(df) - eda_results['Number of offers'], '#E76F51', ',.0f', ''),
         ('Offer rate', eda_results['Percentage offered a place'] * 100, '#7209B7', '.1f', '%')]
for position, (label, value, colour, number_format, suffix) in enumerate(cards):
    fig.add_trace(go.Indicator(mode='number', value=value, title=dict(text=f'<b>{label}</b>', font=dict(size=16)),
                               number=dict(font=dict(color=colour, size=54), valueformat=number_format, suffix=suffix),
                               domain=dict(row=0, column=position)))
fig.update_layout(grid=dict(rows=1, columns=4, pattern='independent'), template='plotly_white', height=230,
                  margin=dict(t=70, b=20), title=dict(text='<b>Number of Records</b>', x=0.01),
                  font=dict(family='Arial, sans-serif'))
show(fig)

donut = go.Figure(go.Pie(labels=['Offered', 'Not offered'],
                         values=[eda_results['Number of offers'], len(df) - eda_results['Number of offers']],
                         hole=0.58, pull=[0.04, 0], marker=dict(colors=[OUTCOME_COLOURS['Offered'], OUTCOME_COLOURS['Not offered']],
                                                                line=dict(color='white', width=3)),
                         textinfo='label+percent+value', textfont=dict(size=15),
                         hovertemplate='%{label}: %{value:,} applicants (%{percent})<extra></extra>'))
style_figure(donut, 'Percentage of students offered a place', 'Share of all applications in the combined dataset', height=480, top_margin=110)
donut.add_annotation(text=f"<b>{eda_results['Percentage offered a place']*100:.1f}%</b><br>offered", showarrow=False, font=dict(size=22))
show(donut)

In [11]:
# Comparison of places offered by gender
gender_table, fig = outcome_by_group('gender', order=['Female', 'Male', 'Other'],
                                     title='Places offered by gender',
                                     subtitle='Left: applicants offered and not offered. Right: offer rate for each gender against the overall rate')
for gender, row in gender_table.iterrows():
    eda_results[f'Applications: gender {gender}'] = int(row['applications'])
    eda_results[f'Offers: gender {gender}'] = int(row['offers'])
    eda_results[f'Offer rate: gender {gender}'] = row['offers'] / row['applications']
show(fig)

display(gender_table.rename(columns={'applications': 'Applications', 'offers': 'Offers', 'not_offered': 'Not offered',
                                     'offer_rate_pct': 'Offer rate (%)'}).round(2))
result = chi_square_vs_outcome(df, 'gender')
print(f"Chi-square test of gender against the outcome: p-value = {result['p-value']:.3f} "
      f"({'a statistically significant difference' if result['p-value'] < 0.05 else 'no statistically significant difference'}).")

,Applications,Offers,Not offered,Offer rate (%)
gender,,,,
Female,4028,2453,1575,60.90
Male,3808,2307,1501,60.58
Other,164,90,74,54.88


Chi-square test of gender against the outcome: p-value = 0.301 (no statistically significant difference).


In [12]:
# Number of applications and overall offer rate by recruiter
recruiter_table = df.groupby('recruiter')[TARGET].agg(applications='size', offers='sum')
recruiter_table['offer_rate_pct'] = recruiter_table['offers'] / recruiter_table['applications'] * 100
for recruiter, row in recruiter_table.iterrows():
    eda_results[f'Applications: {recruiter}'] = int(row['applications'])
    eda_results[f'Offer rate: {recruiter}'] = row['offers'] / row['applications']

fig = make_subplots(rows=1, cols=2, subplot_titles=('Number of applications', 'Overall offer rate (%)'), horizontal_spacing=0.12)
for recruiter, colour in RECRUITER_COLOURS.items():
    row = recruiter_table.loc[recruiter]
    fig.add_trace(go.Bar(x=[recruiter], y=[row['applications']], name=recruiter, marker_color=colour, legendgroup=recruiter,
                         text=[f"{row['applications']:,.0f}"], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}<br>%{y:,} applications<extra></extra>'), row=1, col=1)
    fig.add_trace(go.Bar(x=[recruiter], y=[row['offer_rate_pct']], name=recruiter, marker_color=colour, legendgroup=recruiter,
                         showlegend=False, text=[f"{row['offer_rate_pct']:.1f}%"], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}<br>%{y:.2f}% offered<extra></extra>'), row=1, col=2)
style_figure(fig, 'Applications and offer rate by recruiter', 'Each recruiter supplied the same number of applications; compare the offer rates on the right')
fig.update_yaxes(title_text='Applications', range=[0, recruiter_table['applications'].max() * 1.15], row=1, col=1)
fig.update_yaxes(title_text='Offer rate (%)', range=[0, 100], row=1, col=2)
fig.update_layout(legend=dict(orientation='h', y=-0.15))
add_palette_dropdown(fig, trace_ids=[0, 1, 2, 3], groups=[0, 1, 0, 1])
show(fig)

display(recruiter_table.rename(columns={'applications': 'Applications', 'offers': 'Offers', 'offer_rate_pct': 'Offer rate (%)'}).round(2))
result = chi_square_vs_outcome(df, 'recruiter')
print(f"Chi-square test of recruiter against the outcome: p-value = {result['p-value']:.3f} "
      f"({'the recruiters differ significantly' if result['p-value'] < 0.05 else 'no statistically significant difference between recruiters'}).")

,Applications,Offers,Offer rate (%)
recruiter,,,
Recruiter A,4000,2395,59.88
Recruiter B,4000,2455,61.38


Chi-square test of recruiter against the outcome: p-value = 0.177 (no statistically significant difference between recruiters).


In [13]:
# Mean and median of the three required variables
required_variables = ['prior_attainment_score', 'admissions_test_score', 'distance_from_campus_km']
variable_labels = {'prior_attainment_score': 'Prior attainment score', 'admissions_test_score': 'Admissions test score',
                   'distance_from_campus_km': 'Distance from campus (km)'}
summary = pd.DataFrame({
    'Mean': df[required_variables].mean(), 'Median': df[required_variables].median(),
    'Standard deviation': df[required_variables].std(), 'Minimum': df[required_variables].min(),
    'Maximum': df[required_variables].max(), 'Skewness': df[required_variables].apply(lambda s: skew(s)),
})
for variable in required_variables:
    eda_results[f'Mean: {variable}'] = float(df[variable].mean())
    eda_results[f'Median: {variable}'] = float(df[variable].median())

show(plotly_table(summary.rename(index=variable_labels).reset_index().rename(columns={'index': 'Variable'}),
                  'Descriptive statistics', decimals=2, height=230))

fig = go.Figure()
fig.add_trace(go.Bar(x=[variable_labels[v] for v in required_variables], y=summary['Mean'], name='Mean', marker_color='#3A86FF',
                     text=[f'{v:.1f}' for v in summary['Mean']], textposition='outside', cliponaxis=False,
                     hovertemplate='%{x}<br>Mean: %{y:.2f}<extra></extra>'))
fig.add_trace(go.Bar(x=[variable_labels[v] for v in required_variables], y=summary['Median'], name='Median', marker_color='#FF006E',
                     text=[f'{v:.1f}' for v in summary['Median']], textposition='outside', cliponaxis=False,
                     hovertemplate='%{x}<br>Median: %{y:.2f}<extra></extra>'))
fig.update_layout(barmode='group', legend=dict(orientation='h', y=-0.18))
style_figure(fig, 'Mean and median of the key numeric variables',
             'Where the mean sits well above the median the distribution is skewed to the right (distance from campus)')
fig.update_yaxes(title='Value', range=[0, summary[['Mean', 'Median']].values.max() * 1.18])
add_palette_dropdown(fig, trace_ids=[0, 1])
show(fig)

In [14]:
# Distribution of each required variable: counts per bin, coloured by count, with mean and median marked
def distribution_figure(series, title, unit, bins=30):
    counts, edges = np.histogram(series, bins=bins)
    centres = (edges[:-1] + edges[1:]) / 2
    width = (edges[1] - edges[0]) * 0.94
    fig = go.Figure(go.Bar(
        x=centres, y=counts, width=width, text=counts, textposition='outside', textfont=dict(size=10), cliponaxis=False,
        marker=dict(color=counts, colorscale='Viridis', showscale=True, colorbar=dict(title='Applicants'),
                    line=dict(color='white', width=0.5)),
        customdata=np.stack([edges[:-1], edges[1:]], axis=1),
        hovertemplate=unit + ' %{customdata[0]:.1f} to %{customdata[1]:.1f}<br>Applicants: %{y:,}<extra></extra>'))
    mean_value, median_value = series.mean(), series.median()
    fig.add_vline(x=mean_value, line_dash='dash', line_color='#E63946', line_width=3,
                  annotation_text=f'Mean {mean_value:.1f}', annotation_position='top right', annotation_font_color='#E63946')
    fig.add_vline(x=median_value, line_dash='dot', line_color='#1D3557', line_width=3,
                  annotation_text=f'Median {median_value:.1f}', annotation_position='top left', annotation_font_color='#1D3557')
    style_figure(fig, title, f'Skewness {skew(series):.2f}; red dashed line is the mean, navy dotted line is the median')
    fig.update_xaxes(title=unit)
    fig.update_yaxes(title='Applicants', range=[0, counts.max() * 1.3])
    add_cmap_dropdown(fig, 'marker.colorscale')
    return fig


show(distribution_figure(df['prior_attainment_score'], 'Distribution of prior attainment score', 'Prior attainment score (0 to 100)'))
show(distribution_figure(df['admissions_test_score'], 'Distribution of admissions test score', 'Admissions test score (0 to 100)'))
show(distribution_figure(df['distance_from_campus_km'], 'Distribution of distance from campus', 'Distance from campus (km)'))

In [15]:
# How the three variables differ between applicants who were and were not offered a place
fig = make_subplots(rows=1, cols=3, subplot_titles=[variable_labels[v] for v in required_variables], horizontal_spacing=0.07)
for position, variable in enumerate(required_variables, start=1):
    for outcome, name in OUTCOME_NAMES.items():
        fig.add_trace(go.Box(y=df.loc[df[TARGET] == outcome, variable], name=name, marker_color=OUTCOME_COLOURS[name],
                             boxmean=True, boxpoints='outliers', legendgroup=name, showlegend=(position == 1)),
                      row=1, col=position)
style_figure(fig, 'Offered and not offered applicants compared', 'Boxes show the middle half of values; the dashed line is the mean', height=540)
fig.update_layout(legend=dict(orientation='h', y=-0.12))
add_palette_dropdown(fig, trace_ids=list(range(6)), groups=[0, 1, 0, 1, 0, 1])
show(fig)

In [16]:
# Relationship between prior attainment and the admissions test score
x, y = df['prior_attainment_score'], df['admissions_test_score']
pearson_r, pearson_p = pearsonr(x, y)
spearman_rho, spearman_p = spearmanr(x, y)
slope, intercept = np.polyfit(x, y, 1)
eda_results['Pearson correlation: attainment and test score'] = float(pearson_r)

fig = go.Figure()
for outcome, name in OUTCOME_NAMES.items():
    part = df[df[TARGET] == outcome]
    fig.add_trace(go.Scattergl(x=part['prior_attainment_score'], y=part['admissions_test_score'], mode='markers', name=name,
                               marker=dict(color=OUTCOME_COLOURS[name], size=6, opacity=0.45), text=part.index,
                               hovertemplate='Applicant %{text}<br>Attainment: %{x:.1f}<br>Test score: %{y:.1f}<extra>' + name + '</extra>'))
line_x = np.array([x.min(), x.max()])
fig.add_trace(go.Scatter(x=line_x, y=intercept + slope * line_x, mode='lines', name='Least squares line',
                         line=dict(color='#1D3557', width=4, dash='dash')))
fig.add_annotation(text=f'Pearson r = {pearson_r:.3f} (p = {pearson_p:.3f})<br>Spearman rho = {spearman_rho:.3f}<br>R squared = {pearson_r**2:.4f}',
                   x=0.02, y=0.98, xref='paper', yref='paper', showarrow=False, align='left', xanchor='left', yanchor='top',
                   bgcolor='rgba(255,255,255,0.92)', bordercolor='#ADB5BD', font=dict(size=13))
fig.update_layout(legend=dict(orientation='h', y=-0.18))
style_figure(fig, 'Prior attainment against admissions test score',
             'Each dot is an applicant coloured by outcome; the line shows the overall linear trend')
fig.update_xaxes(title='Prior attainment score'); fig.update_yaxes(title='Admissions test score')
add_palette_dropdown(fig, trace_ids=[0, 1])
show(fig)

strength = 'negligible' if abs(pearson_r) < 0.1 else 'weak' if abs(pearson_r) < 0.3 else 'moderate' if abs(pearson_r) < 0.5 else 'strong'
print(f'The relationship between attainment and the admissions test score is {strength} (Pearson r = {pearson_r:.3f}). '
      'The two measures capture largely different information about an applicant.')

The relationship between attainment and the admissions test score is negligible (Pearson r = 0.015). The two measures capture largely different information about an applicant.


In [17]:
# Counts of applicants in each attainment and test score band, shown as an annotated heatmap
bins = np.linspace(20, 100, 11)
counts, x_edges, y_edges = np.histogram2d(x, y, bins=[bins, bins])
labels = [f'{int(a)} to {int(b)}' for a, b in zip(bins[:-1], bins[1:])]
fig = go.Figure(go.Heatmap(z=counts.T, x=labels, y=labels, colorscale='Viridis', texttemplate='%{z:.0f}',
                           colorbar=dict(title='Applicants'),
                           hovertemplate='Attainment %{x}<br>Test score %{y}<br>Applicants: %{z:.0f}<extra></extra>'))
style_figure(fig, 'Where applicants sit: attainment band against test score band', 'Each cell shows the number of applicants (scores above 100 were capped at 100 during cleaning)', height=560)
fig.update_xaxes(title='Prior attainment score band'); fig.update_yaxes(title='Admissions test score band')
add_cmap_dropdown(fig, 'colorscale')
show(fig)

# Correlation matrix of the numeric variables
numeric_view = df[['prior_attainment_score', 'admissions_test_score', 'personal_statement_score',
                   'distance_from_campus_km', TARGET]]
correlation = numeric_view.corr()
names = ['Prior attainment', 'Admissions test', 'Personal statement', 'Distance', 'Offer (0 or 1)']
fig = go.Figure(go.Heatmap(z=correlation.values, x=names, y=names, colorscale='RdBu', zmid=0, zmin=-1, zmax=1,
                           texttemplate='%{z:.2f}', colorbar=dict(title='Pearson r'),
                           hovertemplate='%{y} and %{x}<br>r = %{z:.3f}<extra></extra>'))
style_figure(fig, 'Correlation between the numeric variables', 'Values near zero mean little linear relationship; attainment and test score are the strongest links to an offer', height=560)
fig.update_yaxes(autorange='reversed')
add_cmap_dropdown(fig, 'colorscale', initial='RdBu')
show(fig)

### Optional analyses

These add context on disability, gender mix, ethnicity and subject choice. They also show whether historical offer rates differ between groups, which matters for the fairness work later.

In [18]:
# Disability distribution and offer rate
disability_order = ['None', 'Learning', 'Mental Health', 'Physical', 'Not Recorded']
disability_table, fig = outcome_by_group('disability_status', order=disability_order,
                                         title='Disability distribution and offers',
                                         subtitle='"None" is a genuine answer. The one applicant with no recorded status is in the table but not charted')
show(fig)
display(disability_table.rename(columns={'applications': 'Applications', 'offers': 'Offers', 'not_offered': 'Not offered',
                                         'offer_rate_pct': 'Offer rate (%)'}).round(2))

# Gender proportion
gender_counts = df['gender'].value_counts().reindex(['Female', 'Male', 'Other'])
fig = go.Figure(go.Pie(labels=gender_counts.index, values=gender_counts.values, hole=0.55,
                       marker=dict(colors=[GENDER_COLOURS[g] for g in gender_counts.index], line=dict(color='white', width=3)),
                       textinfo='label+percent+value', textfont=dict(size=15),
                       hovertemplate='%{label}: %{value:,} applicants (%{percent})<extra></extra>'))
style_figure(fig, 'Gender proportion of applicants', 'Share of the 8,000 applicants by gender', height=480, top_margin=110)
show(fig)

# Age distribution: only possible if the data contains an age column
if 'age' in df.columns:
    show(distribution_figure(df['age'], 'Distribution of age', 'Age (years)'))
else:
    print('Age distribution is not available: the supplied data dictionary has no age column, so none was created or estimated.')

,Applications,Offers,Not offered,Offer rate (%)
disability_status,,,,
None,6209,3768,2441,60.69
Learning,807,478,329,59.23
Mental Health,492,302,190,61.38
Physical,491,302,189,61.51
Not Recorded,1,0,1,0.00


Age distribution is not available: the supplied data dictionary has no age column, so none was created or estimated.


In [19]:
# Places offered by ethnicity
ethnicity_table, fig = outcome_by_group('ethnicity', order=['White', 'Black', 'Asian', 'Mixed', 'Other'],
                                        title='Places offered by ethnicity',
                                        subtitle='Left: applicants offered and not offered. Right: offer rate for each ethnic group against the overall rate')
show(fig)
display(ethnicity_table.rename(columns={'applications': 'Applications', 'offers': 'Offers', 'not_offered': 'Not offered',
                                        'offer_rate_pct': 'Offer rate (%)'}).round(2))
result = chi_square_vs_outcome(df, 'ethnicity')
print(f"Chi-square test of ethnicity against the outcome: p-value = {result['p-value']:.3f} "
      f"({'a statistically significant difference' if result['p-value'] < 0.05 else 'no statistically significant difference'}).")

# Distribution of intended subject area by gender
subject_order = ['STEM', 'Humanities', 'Social Sciences', 'Arts']
subject_by_gender = pd.crosstab(df['gender'], df['intended_subject_area']).reindex(index=['Female', 'Male', 'Other'], columns=subject_order)
fig = go.Figure()
for gender in subject_by_gender.index:
    fig.add_trace(go.Bar(x=subject_order, y=subject_by_gender.loc[gender], name=gender, marker_color=GENDER_COLOURS[gender],
                         text=subject_by_gender.loc[gender], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}<br>' + gender + ': %{y:,}<extra></extra>'))
fig.update_layout(barmode='group', legend=dict(orientation='h', y=-0.18))
style_figure(fig, 'Intended subject area by gender', 'Number of applicants choosing each subject area, split by gender')
fig.update_yaxes(title='Applicants', range=[0, subject_by_gender.values.max() * 1.15])
add_palette_dropdown(fig, trace_ids=[0, 1, 2])
show(fig)

# The same split as a percentage of each gender, which makes the small Other group comparable
subject_share = subject_by_gender.div(subject_by_gender.sum(axis=1), axis=0) * 100
fig = go.Figure(go.Heatmap(z=subject_share.values, x=subject_order, y=subject_share.index, colorscale='Viridis',
                           texttemplate='%{z:.1f}%', colorbar=dict(title='% of gender'),
                           hovertemplate='%{y} applicants choosing %{x}: %{z:.1f}%<extra></extra>'))
style_figure(fig, 'Subject choice as a share of each gender', 'Each row sums to 100 percent', height=460)
add_cmap_dropdown(fig, 'colorscale')
show(fig)

,Applications,Offers,Not offered,Offer rate (%)
ethnicity,,,,
White,1563,924,639,59.12
Black,1593,993,600,62.34
Asian,1597,974,623,60.99
Mixed,1614,985,629,61.03
Other,1633,974,659,59.64


Chi-square test of ethnicity against the outcome: p-value = 0.367 (no statistically significant difference).


## Test the exploratory code

The brief asks for the exploratory code to be tested to confirm that the programme works. Two independent tests are used.

- **Independent Python test.** Every figure stored in `eda_results` is recomputed with plain Python loops and the `statistics` module, without pandas, and compared with the pandas result.
- **Excel test.** The next cells export a workbook in which every figure is also calculated by **live Excel formulas** (`COUNTIFS`, `AVERAGE`, `MEDIAN`, `CORREL`). The workbook compares the Excel result with the Python result and reports `MATCH` or `CHECK`.

In [20]:
EXCEL_PATH = Path('Northbridge_Combined_Admissions_Data.xlsx')
DATA_SHEET = 'Combined_Cleaned'
ARIAL = 'Arial'
HEADER_FILL = PatternFill('solid', start_color='264653')
HEADER_FONT = Font(name=ARIAL, bold=True, color='FFFFFF', size=10)
BODY_FONT = Font(name=ARIAL, size=10)
INPUT_FONT = Font(name=ARIAL, size=10, color='0000FF')


def style_sheet(ws, widths=None, freeze='A2', filter_header=True):
    """Apply the shared professional formatting to a data sheet."""
    for cell in ws[1]:
        cell.fill, cell.font = HEADER_FILL, HEADER_FONT
        cell.alignment = Alignment(horizontal='left', vertical='center', wrap_text=True)
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font = BODY_FONT
    for index, column_cells in enumerate(ws.columns, start=1):
        longest = max(len(str(c.value)) if c.value is not None else 0 for c in list(column_cells)[:300])
        ws.column_dimensions[get_column_letter(index)].width = (widths or {}).get(index, min(max(longest + 3, 12), 60))
    ws.freeze_panes = freeze
    if filter_header:
        ws.auto_filter.ref = ws.dimensions


with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    df.reset_index().to_excel(writer, sheet_name=DATA_SHEET, index=False)
    cleaning_log.to_excel(writer, sheet_name='Cleaning_Log', index=False)
    combined_raw.to_excel(writer, sheet_name='Combined_Raw', index=False)

wb = load_workbook(EXCEL_PATH)
for name in [DATA_SHEET, 'Cleaning_Log', 'Combined_Raw']:
    style_sheet(wb[name])
for letter in ['C', 'D', 'E', 'H']:
    for cell in wb[DATA_SHEET][letter][1:]:
        cell.number_format = '0.00' if letter == 'E' else '0.0'

# Excel_Validation sheet with live formulas
headers = [c.value for c in wb[DATA_SHEET][1]]
column_letter = {name: get_column_letter(i + 1) for i, name in enumerate(headers)}
last_row = len(df) + 1


def rng(name):
    return f'{DATA_SHEET}!${column_letter[name]}$2:${column_letter[name]}${last_row}'


checks = [('Number of records', f'=COUNTA({rng("student_id")})', '#,##0'),
          ('Number of offers', f'=SUM({rng(TARGET)})', '#,##0'),
          ('Percentage offered a place', f'=SUM({rng(TARGET)})/COUNTA({rng("student_id")})', '0.00%')]
for gender in ['Female', 'Male', 'Other']:
    checks += [(f'Applications: gender {gender}', f'=COUNTIF({rng("gender")},"{gender}")', '#,##0'),
               (f'Offers: gender {gender}', f'=COUNTIFS({rng("gender")},"{gender}",{rng(TARGET)},1)', '#,##0'),
               (f'Offer rate: gender {gender}', f'=COUNTIFS({rng("gender")},"{gender}",{rng(TARGET)},1)/COUNTIF({rng("gender")},"{gender}")', '0.00%')]
for recruiter in ['Recruiter A', 'Recruiter B']:
    checks += [(f'Applications: {recruiter}', f'=COUNTIF({rng("recruiter")},"{recruiter}")', '#,##0'),
               (f'Offer rate: {recruiter}', f'=COUNTIFS({rng("recruiter")},"{recruiter}",{rng(TARGET)},1)/COUNTIF({rng("recruiter")},"{recruiter}")', '0.00%')]
for variable in required_variables:
    checks += [(f'Mean: {variable}', f'=AVERAGE({rng(variable)})', '0.0000'),
               (f'Median: {variable}', f'=MEDIAN({rng(variable)})', '0.0000')]
checks.append(('Pearson correlation: attainment and test score',
               f'=CORREL({rng("prior_attainment_score")},{rng("admissions_test_score")})', '0.000000'))
block = f'{DATA_SHEET}!${column_letter["prior_attainment_score"]}$2:${column_letter[TARGET]}${last_row}'
extra = [('Missing cells in the cleaned data columns', f'=COUNTBLANK({block})', '#,##0'),
         ('Scores above 100 in the cleaned data',
          f'=COUNTIF({rng("prior_attainment_score")},">100")+COUNTIF({rng("admissions_test_score")},">100")', '#,##0')]
actual_extra = {'Missing cells in the cleaned data columns': int(df.drop(columns=['recruiter', 'cleaning_flags']).isna().sum().sum()),
                'Scores above 100 in the cleaned data': int((df['prior_attainment_score'] > 100).sum() + (df['admissions_test_score'] > 100).sum())}

ws = wb.create_sheet('Excel_Validation', 1)
ws['A1'] = 'Excel test of the exploratory data analysis'
ws['A1'].font = Font(name=ARIAL, bold=True, size=14, color='264653')
ws['A2'] = ('Column B is calculated by live Excel formulas on the Combined_Cleaned sheet. Column C holds the result '
            'calculated by the Python notebook (typed values, blue). The status shows MATCH when they agree.')
ws['A2'].font = Font(name=ARIAL, size=10, italic=True)
ws['A3'] = 'Summary'
ws['A3'].font = Font(name=ARIAL, bold=True, size=10)
header_row = 5
for position, title in enumerate(['Metric', 'Excel formula result', 'Python result', 'Difference', 'Status'], start=1):
    cell = ws.cell(row=header_row, column=position, value=title)
    cell.fill, cell.font = HEADER_FILL, HEADER_FONT
row = header_row + 1
for label, formula, number_format in checks + extra:
    python_value = eda_results[label] if label in eda_results else actual_extra[label]
    ws.cell(row=row, column=1, value=label).font = BODY_FONT
    ws.cell(row=row, column=2, value=formula).font = BODY_FONT
    ws.cell(row=row, column=3, value=float(python_value)).font = INPUT_FONT
    ws.cell(row=row, column=4, value=f'=B{row}-C{row}').font = BODY_FONT
    ws.cell(row=row, column=5, value=f'=IF(ABS(D{row})<0.000001,"MATCH","CHECK")').font = Font(name=ARIAL, size=10, bold=True)
    for letter in ['B', 'C']:
        ws[f'{letter}{row}'].number_format = number_format
    ws[f'D{row}'].number_format = '0.000000'
    row += 1
final_row = row - 1
ws['B3'] = f'=COUNTIF(E{header_row + 1}:E{final_row},"MATCH")&" of "&COUNTA(A{header_row + 1}:A{final_row})&" checks match"'
ws['B3'].font = Font(name=ARIAL, bold=True, size=10, color='1B7F4B')
from openpyxl.formatting.rule import CellIsRule
ws.conditional_formatting.add(f'E{header_row + 1}:E{final_row}', CellIsRule(operator='equal', formula=['"MATCH"'], fill=PatternFill('solid', start_color='B7E4C7', end_color='B7E4C7')))
ws.conditional_formatting.add(f'E{header_row + 1}:E{final_row}', CellIsRule(operator='equal', formula=['"CHECK"'], fill=PatternFill('solid', start_color='F4A6A6', end_color='F4A6A6')))
for letter, width in zip('ABCDE', [52, 22, 18, 14, 12]):
    ws.column_dimensions[letter].width = width
ws.freeze_panes = f'A{header_row + 1}'

# Read_Me sheet
readme = wb.create_sheet('Read_Me')
lines = [('Northbridge University admissions: combined dataset', True),
         (f'Created by the Python notebook on {datetime.now():%d %B %Y}.', False), ('', False),
         ('Sheets', True),
         ('Combined_Cleaned: recruiter_A.csv and recruiter_B.xlsx merged, validated and cleaned, one row per applicant, indexed on student_id.', False),
         ('Excel_Validation: live formulas that re-calculate the exploratory figures and compare them with the Python results.', False),
         ('Cleaning_Log: every cleaning decision with the number of rows affected.', False),
         ('Combined_Raw: the merged data before any cleaning.', False), ('', False),
         ('Important notes', True),
         ("The value None in disability_status is a genuine answer (no disability). When reading this file with pandas use keep_default_na=False and na_values=[''].", False),
         ('Not Recorded in disability_status marks one applicant whose answer was blank.', False),
         ('recruiter is derived from the student_id prefix. cleaning_flags lists every change made to that applicant record.', False),
         ('No age column exists in the source data.', False), ('', False),
         ('Column dictionary', True)]
dictionary = [('student_id', 'Unique applicant identifier; prefix A or B identifies the recruiter'),
              ('recruiter', 'Recruiter derived from the ID prefix'),
              ('prior_attainment_score', 'Prior academic attainment, 0 to 100'),
              ('admissions_test_score', 'University admissions test score, 0 to 100'),
              ('personal_statement_score', 'Personal statement evaluation, 1 to 5'),
              ('school_type', 'State, Independent or International'),
              ('intended_subject_area', 'STEM, Humanities, Social Sciences or Arts'),
              ('distance_from_campus_km', 'Distance between home and campus in kilometres'),
              ('first_generation_student', '1 if first in family to attend university, otherwise 0'),
              ('gender', 'Male, Female or Other'), ('ethnicity', 'White, Black, Asian, Mixed or Other'),
              ('disability_status', 'None, Physical, Learning, Mental Health or Not Recorded'),
              ('admission_outcome', '1 if an offer was made, otherwise 0'),
              ('cleaning_flags', 'Semicolon separated list of cleaning actions applied to the record')]
for position, (text, bold) in enumerate(lines, start=1):
    readme.cell(row=position, column=1, value=text).font = Font(name=ARIAL, bold=bold, size=12 if bold else 10)
start = len(lines) + 1
for offset, (name, meaning) in enumerate(dictionary):
    readme.cell(row=start + offset, column=1, value=name).font = Font(name=ARIAL, bold=True, size=10)
    readme.cell(row=start + offset, column=2, value=meaning).font = BODY_FONT
readme.column_dimensions['A'].width = 38
readme.column_dimensions['B'].width = 80
wb.active = 0
wb.save(EXCEL_PATH)
print(f'Saved {EXCEL_PATH} with sheets: {", ".join(wb.sheetnames)}')
print(f'Excel_Validation contains {len(checks) + len(extra)} live-formula checks; open the file in Excel to see MATCH for each one.')

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(EXCEL_PATH))
    except Exception as error:
        print('Download the workbook from the Colab files panel:', error)

Saved Northbridge_Combined_Admissions_Data.xlsx with sheets: Combined_Cleaned, Excel_Validation, Cleaning_Log, Combined_Raw, Read_Me
Excel_Validation contains 25 live-formula checks; open the file in Excel to see MATCH for each one.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Modelling: design decisions

**Target.** `admission_outcome` (1 offer, 0 no offer).

**Models.** Two classifiers were specified: **Naive Bayes** and **XGBoost**. Both are compared against a **majority-class baseline** (always predict the most common outcome). A model that cannot beat this baseline has learned nothing useful.

**Loss function.** The primary measure is **log loss** (binary cross-entropy), the loss that both models optimise. It scores the quality of the predicted probabilities, not just the final yes or no, which matters for a shortlisting tool. Accuracy, balanced accuracy, precision, recall, F1, ROC AUC, PR AUC, Brier score and Matthews correlation are reported as well.

**Data split.** The data is split into training (70 percent), validation (15 percent) and test (15 percent) sets, stratified on the outcome. The training set is used for fitting and tuning, the validation set for early stopping and model selection, and the test set only for the final unbiased check.

In [21]:
# Reusable preprocessing: scale numbers and one-hot encode categories, learned from the training data only
XGB_BASE = dict(objective='binary:logistic', eval_metric='logloss', tree_method='hist', random_state=SEED, n_jobs=1, verbosity=0)


def build_preprocessor(features):
    """Return a fresh, unfitted preprocessing step for the chosen features."""
    numeric = [c for c in features if c in NUMERIC_COLUMNS]
    categorical = [c for c in features if c not in numeric]
    preprocessor = ColumnTransformer([('numeric', StandardScaler(), numeric),
                                      ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical)])
    preprocessor.set_output(transform='pandas')
    return preprocessor


PROTECTED = ['gender', 'ethnicity', 'disability_status', 'first_generation_student']
PROXY_RISK = ['school_type', 'distance_from_campus_km']
GOVERNED = SCORE_COLUMNS + ['intended_subject_area']
FEATURE_SETS = {
    'Governed': GOVERNED,
    'Governed + proxy-risk': GOVERNED + PROXY_RISK,
    'All incl. protected (audit only)': GOVERNED + PROXY_RISK + PROTECTED,
}
SELECTED_SET = 'Governed'
FEATURES = FEATURE_SETS[SELECTED_SET]

X_all = df.drop(columns=[TARGET, 'cleaning_flags'])
y_all = df[TARGET]

# Stratified split on the outcome: 70 percent train, 15 percent validation, 15 percent test
idx_train_val, idx_test = train_test_split(df.index, test_size=0.15, stratify=y_all, random_state=SEED)
idx_train, idx_val = train_test_split(idx_train_val, test_size=0.15 / 0.85, stratify=y_all.loc[idx_train_val], random_state=SEED)
X_train, y_train = X_all.loc[idx_train], y_all.loc[idx_train]
X_val, y_val = X_all.loc[idx_val], y_all.loc[idx_val]
X_test, y_test = X_all.loc[idx_test], y_all.loc[idx_test]

split_table = pd.DataFrame({
    'Split': ['Training', 'Validation', 'Test'],
    'Applicants': [len(idx_train), len(idx_val), len(idx_test)],
    'Offer rate (%)': [y_train.mean() * 100, y_val.mean() * 100, y_test.mean() * 100],
    'Recruiter A share (%)': [(X_train['recruiter'] == 'Recruiter A').mean() * 100, (X_val['recruiter'] == 'Recruiter A').mean() * 100,
                              (X_test['recruiter'] == 'Recruiter A').mean() * 100],
})
split_table['Share of data (%)'] = split_table['Applicants'] / len(df) * 100

fig = go.Figure()
for name, colour in zip(split_table['Split'], ['#264653', '#2A9D8F', '#E9C46A']):
    row = split_table[split_table['Split'] == name].iloc[0]
    fig.add_trace(go.Bar(x=[name], y=[row['Applicants']], name=name, marker_color=colour,
                         text=[f"{row['Applicants']:,.0f}<br>offer rate {row['Offer rate (%)']:.1f}%"], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}: %{y:,} applicants<extra></extra>'))
style_figure(fig, 'Training, validation and test split', 'Stratification keeps the offer rate almost identical in every split')
fig.update_yaxes(title='Applicants', range=[0, split_table['Applicants'].max() * 1.2])
fig.update_layout(showlegend=False)
add_palette_dropdown(fig, trace_ids=[0, 1, 2])
show(fig)
display(split_table.round(2))
print('Selected feature set for the models:', ', '.join(FEATURES))

,Split,Applicants,Offer rate (%),Recruiter A share (%),Share of data (%)
0,Training,5600,60.62,50.52,70.0
1,Validation,1200,60.67,50.25,15.0
2,Test,1200,60.58,47.33,15.0


Selected feature set for the models: prior_attainment_score, admissions_test_score, personal_statement_score, intended_subject_area


## Proxy variable audit

Removing protected characteristics does not guarantee fairness, because the remaining variables may still **encode** them. Two checks are run before any model is trained.

- **Association strength.** How strongly is each allowed or proxy-risk variable associated with each protected characteristic? Cramer's V is used for two categorical variables and the correlation ratio (eta) for a numeric variable against a category. Both run from 0 (no association) to 1 (perfect association).
- **Predictability.** Can a simple model **predict** each protected characteristic from the variables? A cross-validated ROC AUC of 0.50 is chance level, meaning no proxy signal. Values well above 0.50 would show that the variables reveal the protected characteristic.

In [22]:
def cramers_v_between(a, b):
    """Bias-corrected Cramer's V between two categorical series."""
    table = pd.crosstab(a, b)
    chi2 = chi2_contingency(table, correction=False)[0]
    n = table.values.sum()
    r, k = table.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    r_corr, k_corr = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    denominator = min(k_corr - 1, r_corr - 1)
    return math.sqrt(phi2 / denominator) if denominator > 0 else 0.0


def correlation_ratio(categories, values):
    """Correlation ratio (eta): how much of a numeric variable's variance is explained by a category."""
    overall = values.mean()
    between = sum(len(g) * (g.mean() - overall) ** 2 for _, g in values.groupby(categories))
    total = ((values - overall) ** 2).sum()
    return math.sqrt(between / total) if total > 0 else 0.0


audit_data = df[df['disability_status'] != 'Not Recorded'].copy()
audit_data['first_generation_student'] = audit_data['first_generation_student'].map({0: 'No', 1: 'Yes'})
candidates = {'Prior attainment score': ('numeric', 'prior_attainment_score'), 'Admissions test score': ('numeric', 'admissions_test_score'),
              'Personal statement score': ('numeric', 'personal_statement_score'), 'Intended subject area': ('category', 'intended_subject_area'),
              'School type (proxy risk)': ('category', 'school_type'), 'Distance from campus (proxy risk)': ('numeric', 'distance_from_campus_km')}
protected_labels = {'gender': 'Gender', 'ethnicity': 'Ethnicity', 'disability_status': 'Disability', 'first_generation_student': 'First generation'}

association_matrix = pd.DataFrame(index=list(candidates), columns=list(protected_labels.values()), dtype=float)
for label, (kind, column) in candidates.items():
    for protected, protected_label in protected_labels.items():
        association_matrix.loc[label, protected_label] = (correlation_ratio(audit_data[protected], audit_data[column]) if kind == 'numeric'
                                                          else cramers_v_between(audit_data[column], audit_data[protected]))

fig = go.Figure(go.Heatmap(z=association_matrix.values, x=association_matrix.columns, y=association_matrix.index, colorscale='Viridis',
                           zmin=0, zmax=max(0.2, float(association_matrix.values.max())), texttemplate='%{z:.3f}',
                           colorbar=dict(title='Association'), hovertemplate='%{y} and %{x}<br>Association: %{z:.3f}<extra></extra>'))
style_figure(fig, 'Association between allowed variables and protected characteristics',
             'Values near zero mean the variable carries little information about the protected characteristic', height=560)
fig.update_yaxes(autorange='reversed')
add_cmap_dropdown(fig, 'colorscale')
show(fig)
print(f'Strongest association found: {association_matrix.values.max():.3f} (scale 0 to 1).')

Strongest association found: 0.032 (scale 0 to 1).


In [23]:
# Predictability test: can the variables reveal a protected characteristic?
proxy_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
predict_rows = []
for protected, protected_label in protected_labels.items():
    target_series = audit_data[protected]
    scoring = 'roc_auc' if target_series.nunique() == 2 else 'roc_auc_ovr_weighted'
    row = {'Protected characteristic': protected_label}
    for label, features in [('AUC from governed variables', GOVERNED), ('AUC with proxy-risk variables added', GOVERNED + PROXY_RISK)]:
        model = Pipeline([('prep', build_preprocessor(features)), ('model', LogisticRegression(max_iter=500))])
        scores = cross_validate(model, audit_data[features], target_series, cv=proxy_cv, scoring=scoring)['test_score']
        row[label] = float(scores.mean())
    predict_rows.append(row)

predictability = pd.DataFrame(predict_rows)
predictability['Chance level'] = 0.5
highest = predictability[['AUC from governed variables', 'AUC with proxy-risk variables added']].max(axis=1)
predictability['Reading'] = np.where(highest < 0.55, 'Pass', 'Review')
show(plotly_table(predictability, 'Can the variables predict a protected characteristic? (cross-validated ROC AUC)',
                  status_col='Reading', decimals=3, height=250))

fig = go.Figure()
for label, colour in [('AUC from governed variables', '#2A9D8F'), ('AUC with proxy-risk variables added', '#E76F51')]:
    fig.add_trace(go.Bar(x=predictability['Protected characteristic'], y=predictability[label], name=label.replace('AUC ', ''),
                         marker_color=colour, text=[f'{v:.3f}' for v in predictability[label]], textposition='outside', cliponaxis=False,
                         hovertemplate='%{x}<br>AUC %{y:.3f}<extra></extra>'))
fig.add_trace(go.Scatter(x=predictability['Protected characteristic'], y=[0.5] * len(predictability), mode='lines', name='Chance level (0.50): no proxy signal',
                         line=dict(color='#1D3557', dash='dash', width=3), hoverinfo='skip'))
fig.update_layout(barmode='group', legend=dict(orientation='h', y=-0.18))
style_figure(fig, 'Proxy test: how well can protected characteristics be predicted?',
             'Bars close to the dashed line mean the variables reveal nothing about the protected characteristic')
fig.update_yaxes(title='Cross-validated ROC AUC', range=[0.4, max(0.7, float(highest.max()) + 0.05)])
add_palette_dropdown(fig, trace_ids=[0, 1])
show(fig)
proxy_verdict = ('No variable allows a protected characteristic to be predicted better than chance level, so no strong proxy signal was found.'
                 if (predictability['Reading'] == 'Pass').all() else
                 'At least one protected characteristic can be predicted above chance level: review the variables flagged above.')
print(proxy_verdict)

No variable allows a protected characteristic to be predicted better than chance level, so no strong proxy signal was found.


## Do the extra variables earn their place?

Before tuning, a quick cross-validation on the **training data only** compares the three feature sets with both models. If protected or proxy-risk variables added real predictive value there would be a trade-off to discuss. If they add nothing, excluding them costs nothing.

In [24]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
LOSS_AXIS_CAP = 0.74    # log loss bars above this are drawn at the cap and labelled 'off scale'
ablation_rows = []
for set_name, features in FEATURE_SETS.items():
    candidates_models = {
        'Naive Bayes': Pipeline([('prep', build_preprocessor(features)), ('model', GaussianNB())]),
        'XGBoost': Pipeline([('prep', build_preprocessor(features)),
                             ('model', XGBClassifier(**XGB_BASE, n_estimators=150, max_depth=2, learning_rate=0.05, subsample=0.8,
                                                     colsample_bytree=0.8, min_child_weight=10))]),
    }
    for model_name, model in candidates_models.items():
        scores = cross_validate(model, X_train[features], y_train, cv=cv, scoring={'logloss': 'neg_log_loss', 'auc': 'roc_auc'})
        ablation_rows.append({'Feature set': set_name, 'Model': model_name, 'Log loss': -scores['test_logloss'].mean(),
                              'ROC AUC': scores['test_auc'].mean()})
ablation = pd.DataFrame(ablation_rows)

fig = make_subplots(rows=1, cols=2, subplot_titles=('ROC AUC (higher is better)', 'Log loss (lower is better)'), horizontal_spacing=0.1)
for model_name in ['Naive Bayes', 'XGBoost']:
    part = ablation[ablation['Model'] == model_name]
    fig.add_trace(go.Bar(x=part['Feature set'], y=part['ROC AUC'], name=model_name, marker_color=MODEL_COLOURS[model_name], legendgroup=model_name,
                         text=[f'{v:.3f}' for v in part['ROC AUC']], textposition='outside', cliponaxis=False), row=1, col=1)
    shown_loss = part['Log loss'].clip(upper=LOSS_AXIS_CAP)
    labels_loss = [f'{v:.4f}' if v <= LOSS_AXIS_CAP else f'{v:.2f} (off scale)' for v in part['Log loss']]
    fig.add_trace(go.Bar(x=part['Feature set'], y=shown_loss, name=model_name, marker_color=MODEL_COLOURS[model_name], legendgroup=model_name,
                         showlegend=False, text=labels_loss, textposition='outside', cliponaxis=False, customdata=part['Log loss'],
                         hovertemplate='%{x}<br>Log loss %{customdata:.4f}<extra>' + model_name + '</extra>'), row=1, col=2)
fig.update_layout(barmode='group', legend=dict(orientation='h', y=-0.3))
style_figure(fig, 'Feature set comparison (5-fold cross-validation on training data)',
             'Adding proxy-risk or protected variables does not improve performance (Naive Bayes in particular gets worse)', height=560)
fig.update_yaxes(range=[ablation['ROC AUC'].min() - 0.03, ablation['ROC AUC'].max() + 0.02], row=1, col=1)
fig.update_yaxes(range=[ablation['Log loss'].min() - 0.01, LOSS_AXIS_CAP + 0.012], row=1, col=2)
add_palette_dropdown(fig, trace_ids=[0, 1, 2, 3], groups=[0, 0, 1, 1])
show(fig)

governed_auc = ablation[ablation['Feature set'] == 'Governed']['ROC AUC'].mean()
proxy_auc = ablation[ablation['Feature set'] == 'Governed + proxy-risk']['ROC AUC'].mean()
all_auc = ablation[ablation['Feature set'] == 'All incl. protected (audit only)']['ROC AUC'].mean()
ablation_verdict = (f'Average ROC AUC across both models: governed {governed_auc:.3f}, with proxy-risk variables {proxy_auc:.3f}, '
                    f'with all variables including protected characteristics {all_auc:.3f}. ')
ablation_verdict += ('The extra variables do not improve predictive performance (gain under 0.005 ROC AUC), so excluding them costs nothing.'
                     if max(proxy_auc, all_auc) - governed_auc < 0.005 else
                     'The extra variables add some predictive value, which creates a genuine accuracy and fairness trade-off to discuss.')
nb_governed = ablation[(ablation['Feature set'] == 'Governed') & (ablation['Model'] == 'Naive Bayes')]['Log loss'].iloc[0]
nb_all = ablation[(ablation['Feature set'] == 'All incl. protected (audit only)') & (ablation['Model'] == 'Naive Bayes')]['Log loss'].iloc[0]
if nb_all > nb_governed + 0.05:
    ablation_verdict += (f' Naive Bayes also becomes badly overconfident when many 0 or 1 category columns are added (log loss {nb_all:.2f} against {nb_governed:.4f}), '
                         'because it treats each column as an independent normal distribution. This is a further reason to keep its feature set small.')
print(ablation_verdict)
display(ablation.round(4))

Average ROC AUC across both models: governed 0.572, with proxy-risk variables 0.564, with all variables including protected characteristics 0.553. The extra variables do not improve predictive performance (gain under 0.005 ROC AUC), so excluding them costs nothing. Naive Bayes also becomes badly overconfident when many 0 or 1 category columns are added (log loss 2.37 against 0.6629), because it treats each column as an independent normal distribution. This is a further reason to keep its feature set small.


,Feature set,Model,Log loss,ROC AUC
0,Governed,Naive Bayes,0.6629,0.5755
1,Governed,XGBoost,0.6643,0.5692
2,Governed + proxy-risk,Naive Bayes,0.6680,0.5631
3,Governed + proxy-risk,XGBoost,0.6655,0.5659
4,All incl. protected (audit only),Naive Bayes,2.3723,0.5401
5,All incl. protected (audit only),XGBoost,0.6649,0.5669


## Baseline and model training

In [25]:
# Majority-class baseline
baseline = DummyClassifier(strategy='prior').fit(X_train[FEATURES], y_train)

# Naive Bayes: tune var_smoothing with cross-validated log loss
nb_pipeline = Pipeline([('prep', build_preprocessor(FEATURES)), ('model', GaussianNB())])
nb_search = GridSearchCV(nb_pipeline, {'model__var_smoothing': np.logspace(-9, 0, 19)}, scoring='neg_log_loss',
                         cv=cv, n_jobs=-1, return_train_score=True)
nb_search.fit(X_train[FEATURES], y_train)
nb_model = nb_search.best_estimator_
print(f"Naive Bayes best var_smoothing: {nb_search.best_params_['model__var_smoothing']:.2e} | "
      f"cross-validated log loss: {-nb_search.best_score_:.4f}")

results = pd.DataFrame(nb_search.cv_results_)
smoothing = results['param_model__var_smoothing'].astype(float)
log_loss_mean, log_loss_std = -results['mean_test_score'], results['std_test_score']
best_position = int(np.argmin(log_loss_mean.values))
fig = go.Figure()
fig.add_trace(go.Scatter(x=smoothing, y=log_loss_mean, mode='lines', line=dict(color='#ADB5BD', width=2), showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=smoothing, y=log_loss_mean, mode='markers+text', name='Cross-validated log loss',
                         error_y=dict(type='data', array=log_loss_std, color='#868E96', thickness=1.2),
                         marker=dict(size=14, color=log_loss_mean, colorscale='Viridis', showscale=True, colorbar=dict(title='Log loss'),
                                     line=dict(color='white', width=1)),
                         text=[f'{v:.4f}' if i % 3 == 0 or i == best_position else '' for i, v in enumerate(log_loss_mean)],
                         textposition='top center', hovertemplate='var_smoothing %{x:.1e}<br>Log loss %{y:.4f}<extra></extra>'))
fig.add_trace(go.Scatter(x=[smoothing.iloc[best_position]], y=[log_loss_mean.iloc[best_position]], mode='markers', name='Best setting',
                         marker=dict(size=22, symbol='star', color='#E63946', line=dict(color='white', width=2))))
fig.update_layout(legend=dict(orientation='h', y=-0.2))
style_figure(fig, 'Naive Bayes tuning: log loss against var_smoothing', 'Lower is better; error bars show variation across the five validation folds')
fig.update_xaxes(type='log', title='var_smoothing (log scale)'); fig.update_yaxes(title='Cross-validated log loss')
add_cmap_dropdown(fig, 'marker.colorscale', trace_ids=[1])
show(fig)

Naive Bayes best var_smoothing: 3.16e-01 | cross-validated log loss: 0.6613


In [26]:
# XGBoost: randomised search over regularised settings
xgb_pipeline = Pipeline([('prep', build_preprocessor(FEATURES)), ('model', XGBClassifier(**XGB_BASE))])
xgb_space = {
    'model__n_estimators': randint(50, 400), 'model__learning_rate': [0.01, 0.02, 0.05, 0.1],
    'model__max_depth': [1, 2, 3, 4], 'model__min_child_weight': [1, 5, 10, 20, 40],
    'model__subsample': uniform(0.6, 0.4), 'model__colsample_bytree': uniform(0.6, 0.4),
    'model__reg_lambda': [1, 5, 10, 20, 50], 'model__gamma': [0, 0.5, 1, 2],
}
xgb_search = RandomizedSearchCV(xgb_pipeline, xgb_space, n_iter=30, scoring='neg_log_loss', cv=cv, n_jobs=-1,
                                random_state=SEED, return_train_score=True)
xgb_search.fit(X_train[FEATURES], y_train)
print(f'XGBoost best cross-validated log loss: {-xgb_search.best_score_:.4f}')

trials = pd.DataFrame(xgb_search.cv_results_).sort_values('rank_test_score').head(8)
trial_table = pd.DataFrame({
    'Rank': trials['rank_test_score'].astype(int), 'CV log loss': -trials['mean_test_score'], 'Std': trials['std_test_score'],
    'Trees': trials['param_model__n_estimators'].astype(int), 'Depth': trials['param_model__max_depth'].astype(int),
    'Learning rate': trials['param_model__learning_rate'].astype(float), 'Min child weight': trials['param_model__min_child_weight'].astype(int),
    'Lambda': trials['param_model__reg_lambda'].astype(int), 'Gamma': trials['param_model__gamma'].astype(float)})
show(plotly_table(trial_table, 'Best eight XGBoost settings found by cross-validation', decimals=4, height=380))

# Final XGBoost: best settings, with the number of trees chosen by early stopping on the validation set
best_params = {k.split('__', 1)[1]: v for k, v in xgb_search.best_params_.items()}
preprocessor_xgb = build_preprocessor(FEATURES).fit(X_train[FEATURES])
train_encoded, validation_encoded = preprocessor_xgb.transform(X_train[FEATURES]), preprocessor_xgb.transform(X_val[FEATURES])
final_params = {**XGB_BASE, **{k: v for k, v in best_params.items() if k != 'n_estimators'},
                'n_estimators': 2000, 'early_stopping_rounds': 50}
final_xgb = XGBClassifier(**final_params)
final_xgb.fit(train_encoded, y_train, eval_set=[(train_encoded, y_train), (validation_encoded, y_val)], verbose=False)
best_round = int(final_xgb.best_iteration) + 1
xgb_model = Pipeline([('prep', preprocessor_xgb), ('model', final_xgb)])
print(f'Early stopping chose {best_round} trees (depth {best_params["max_depth"]}, learning rate {best_params["learning_rate"]}).')

history = final_xgb.evals_result()
train_curve, validation_curve = history['validation_0']['logloss'], history['validation_1']['logloss']
baseline_loss = log_loss(y_val, baseline.predict_proba(X_val[FEATURES])[:, 1])
rounds = list(range(1, len(train_curve) + 1))
fig = go.Figure()
fig.add_trace(go.Scatter(x=rounds, y=train_curve, mode='lines', name='Training log loss', line=dict(color='#2A9D8F', width=3)))
fig.add_trace(go.Scatter(x=rounds, y=validation_curve, mode='lines', name='Validation log loss', line=dict(color='#E76F51', width=3)))
fig.add_hline(y=baseline_loss, line_dash='dot', line_color='#8D99AE', annotation_text=f'Majority baseline {baseline_loss:.4f}', annotation_position='top right')
fig.add_vline(x=best_round, line_dash='dash', line_color='#1D3557', annotation_text=f'Early stop at {best_round} trees', annotation_position='bottom left')
fig.update_layout(legend=dict(orientation='h', y=-0.2))
style_figure(fig, 'XGBoost learning curve', 'Training loss keeps falling while validation loss stops improving: the signature of fitting noise')
fig.update_xaxes(title='Number of trees')
fig.update_yaxes(title='Log loss', range=[min(train_curve) - 0.003, baseline_loss + 0.002])
add_palette_dropdown(fig, trace_ids=[0, 1], line=True)
show(fig)

XGBoost best cross-validated log loss: 0.6621


Early stopping chose 511 trees (depth 1, learning rate 0.02).


## Model evaluation on the validation and test sets

In [27]:
def compute_metrics(model, X, y, threshold=0.5):
    """All evaluation metrics for one model on one dataset (decisions use the given probability threshold)."""
    probability = model.predict_proba(X[FEATURES])[:, 1]
    decision = (probability >= threshold).astype(int)
    return {'Log loss': log_loss(y, probability), 'Brier score': brier_score_loss(y, probability),
            'Accuracy': accuracy_score(y, decision), 'Balanced accuracy': balanced_accuracy_score(y, decision),
            'Precision': precision_score(y, decision, zero_division=0), 'Recall': recall_score(y, decision, zero_division=0),
            'F1 score': f1_score(y, decision, zero_division=0), 'ROC AUC': roc_auc_score(y, probability),
            'PR AUC': average_precision_score(y, probability), 'Matthews correlation': matthews_corrcoef(y, decision)}


models = {'Majority baseline': baseline, 'Naive Bayes': nb_model, 'XGBoost': xgb_model}
metric_rows = []
for split_name, (X_part, y_part) in {'Validation': (X_val, y_val), 'Test': (X_test, y_test)}.items():
    for model_name, model in models.items():
        metric_rows.append({'Model': model_name, 'Split': split_name, **compute_metrics(model, X_part, y_part)})
metrics_all = pd.DataFrame(metric_rows)
metrics_test = metrics_all[metrics_all['Split'] == 'Test'].set_index('Model').drop(columns='Split')
metrics_validation = metrics_all[metrics_all['Split'] == 'Validation'].set_index('Model').drop(columns='Split')

show(plotly_table(metrics_all, 'Summary table of accuracy metrics (decision threshold 0.5)', decimals=4, height=300))

# Scorecard heatmap: colour = rank within the metric (1 is best), label = actual value
lower_is_better = {'Log loss', 'Brier score'}
rank_score = pd.DataFrame(index=metrics_test.index, columns=metrics_test.columns, dtype=float)
for metric in metrics_test.columns:
    values = metrics_test[metric]
    span = values.max() - values.min()
    scaled = (values - values.min()) / span if span > 1e-12 else pd.Series(0.5, index=values.index)
    rank_score[metric] = 1 - scaled if metric in lower_is_better else scaled
fig = go.Figure(go.Heatmap(z=rank_score.T.values, x=list(rank_score.index), y=list(rank_score.columns),
                           text=[[f'{metrics_test.loc[m, c]:.4f}' for m in metrics_test.index] for c in metrics_test.columns],
                           texttemplate='%{text}', colorscale='RdYlGn', zmin=0, zmax=1,
                           colorbar=dict(title='Relative rank<br>(1 = best)'),
                           hovertemplate='%{x}<br>%{y}: %{text}<extra></extra>'))
style_figure(fig, 'Test-set scorecard', 'Colour shows relative rank within each metric; the label shows the value', height=640)
fig.update_yaxes(autorange='reversed')
add_cmap_dropdown(fig, 'colorscale', initial='RdYlGn')
show(fig)

In [28]:
# Confusion matrices on the test set
fig = make_subplots(rows=1, cols=2, subplot_titles=('Naive Bayes', 'XGBoost'), horizontal_spacing=0.14)
for position, model_name in enumerate(['Naive Bayes', 'XGBoost'], start=1):
    decision = (models[model_name].predict_proba(X_test[FEATURES])[:, 1] >= 0.5).astype(int)
    matrix = confusion_matrix(y_test, decision)
    row_share = matrix / matrix.sum(axis=1, keepdims=True) * 100
    labels = [[f'{matrix[r, c]:,}<br>{row_share[r, c]:.1f}% of row' for c in range(2)] for r in range(2)]
    fig.add_trace(go.Heatmap(z=matrix, x=['No offer', 'Offer'], y=['No offer', 'Offer'], text=labels,
                             texttemplate='%{text}', colorscale='Viridis', showscale=(position == 2), colorbar=dict(title='Applicants'),
                             hovertemplate='Actual: %{y}<br>Predicted: %{x}<br>Applicants: %{z:,}<extra></extra>'), row=1, col=position)
style_figure(fig, 'Confusion matrices on the test set', 'Each cell shows the number of applicants and the share of that actual outcome', height=560)
fig.update_layout(margin=dict(l=110))
fig.update_yaxes(autorange='reversed', title_text='Actual outcome')
fig.update_xaxes(title_text='Predicted outcome')
add_cmap_dropdown(fig, 'colorscale', trace_ids=[0, 1])
show(fig)

# ROC and precision-recall curves
fig = make_subplots(rows=1, cols=2, subplot_titles=('ROC curve', 'Precision-recall curve'), horizontal_spacing=0.1)
for model_name in ['Naive Bayes', 'XGBoost']:
    probability = models[model_name].predict_proba(X_test[FEATURES])[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probability)
    precision, recall, _ = precision_recall_curve(y_test, probability)
    fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'{model_name} (AUC {roc_auc_score(y_test, probability):.3f})',
                             line=dict(color=MODEL_COLOURS[model_name], width=3), legendgroup=model_name,
                             hovertemplate='False positive rate %{x:.2f}<br>True positive rate %{y:.2f}<extra></extra>'), row=1, col=1)
    fig.add_trace(go.Scatter(x=recall, y=precision, mode='lines', name=model_name, showlegend=False,
                             line=dict(color=MODEL_COLOURS[model_name], width=3), legendgroup=model_name,
                             hovertemplate='Recall %{x:.2f}<br>Precision %{y:.2f}<extra></extra>'), row=1, col=2)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Chance (AUC 0.500)', line=dict(color='#8D99AE', dash='dash', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=[0, 1], y=[y_test.mean()] * 2, mode='lines', name='Majority baseline', line=dict(color='#8D99AE', dash='dot', width=2)), row=1, col=2)
style_figure(fig, 'ROC and precision-recall curves on the test set', 'Curves hugging the dashed baseline show little ability to separate offers from non-offers', height=560)
fig.update_xaxes(title_text='False positive rate', row=1, col=1); fig.update_yaxes(title_text='True positive rate', row=1, col=1)
fig.update_xaxes(title_text='Recall', row=1, col=2); fig.update_yaxes(title_text='Precision', range=[0, 1.02], row=1, col=2)
fig.update_layout(legend=dict(orientation='h', y=-0.2))
add_palette_dropdown(fig, trace_ids=[0, 1, 2, 3], groups=[0, 0, 1, 1], line=True)
show(fig)

In [29]:
# Calibration: do predicted probabilities match the observed offer rates?
fig = go.Figure()
for model_name in ['Naive Bayes', 'XGBoost']:
    probability = models[model_name].predict_proba(X_test[FEATURES])[:, 1]
    observed, predicted = calibration_curve(y_test, probability, n_bins=8, strategy='quantile')
    fig.add_trace(go.Scatter(x=predicted, y=observed, mode='lines+markers+text', name=model_name,
                             line=dict(color=MODEL_COLOURS[model_name], width=3), marker=dict(size=11),
                             text=[f'{v:.2f}' for v in observed], textposition='top center',
                             hovertemplate='Predicted %{x:.3f}<br>Observed %{y:.3f}<extra></extra>'))
fig.add_trace(go.Scatter(x=[0.3, 0.9], y=[0.3, 0.9], mode='lines', name='Perfect calibration', line=dict(color='#8D99AE', dash='dash', width=2)))
fig.update_layout(legend=dict(orientation='h', y=-0.2))
style_figure(fig, 'Calibration of predicted probabilities (test set)', 'Points on the dashed line mean a predicted 60 percent really is a 60 percent chance')
fig.update_xaxes(title='Mean predicted probability of an offer'); fig.update_yaxes(title='Observed share of offers')
add_palette_dropdown(fig, trace_ids=[0, 1], line=True)
show(fig)

# Spread of predicted probabilities: how decisive is each model?
fig = go.Figure()
for model_name in ['Naive Bayes', 'XGBoost']:
    probability = models[model_name].predict_proba(X_test[FEATURES])[:, 1]
    fig.add_trace(go.Histogram(x=probability, name=model_name, marker_color=MODEL_COLOURS[model_name], opacity=0.65, nbinsx=30,
                               hovertemplate='Predicted probability %{x:.2f}<br>Applicants %{y}<extra></extra>'))
fig.update_layout(barmode='overlay', legend=dict(orientation='h', y=-0.2))
style_figure(fig, 'Spread of predicted offer probabilities (test set)',
             'A narrow spread around the base rate means the model cannot confidently separate applicants')
fig.update_xaxes(title='Predicted probability of an offer'); fig.update_yaxes(title='Applicants')
add_palette_dropdown(fig, trace_ids=[0, 1])
show(fig)

In [30]:
def per_row_log_loss(y, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


y_test_array = y_test.values
probability_nb = nb_model.predict_proba(X_test[FEATURES])[:, 1]
probability_xgb = xgb_model.predict_proba(X_test[FEATURES])[:, 1]
loss_nb, loss_xgb = per_row_log_loss(y_test_array, probability_nb), per_row_log_loss(y_test_array, probability_xgb)

generator = np.random.default_rng(SEED)
boot = {'ll_nb': [], 'll_xgb': [], 'auc_nb': [], 'auc_xgb': []}
for _ in range(2000):
    index = generator.integers(0, len(y_test_array), len(y_test_array))
    boot['ll_nb'].append(loss_nb[index].mean()); boot['ll_xgb'].append(loss_xgb[index].mean())
    boot['auc_nb'].append(roc_auc_score(y_test_array[index], probability_nb[index]))
    boot['auc_xgb'].append(roc_auc_score(y_test_array[index], probability_xgb[index]))
boot = {k: np.array(v) for k, v in boot.items()}


def interval(values):
    return np.percentile(values, [2.5, 97.5])


ll_difference = boot['ll_nb'] - boot['ll_xgb']          # positive means XGBoost has the lower loss
auc_difference = boot['auc_xgb'] - boot['auc_nb']       # positive means XGBoost has the higher AUC
ll_low, ll_high = interval(ll_difference)
auc_low, auc_high = interval(auc_difference)


def verdict(low, high, positive_name, negative_name):
    if low > 0:
        return f'{positive_name} is clearly better'
    if high < 0:
        return f'{negative_name} is clearly better'
    return 'No clear difference'


bootstrap_table = pd.DataFrame([
    {'Metric': 'Log loss', 'Naive Bayes': f"{loss_nb.mean():.4f} [{interval(boot['ll_nb'])[0]:.4f}, {interval(boot['ll_nb'])[1]:.4f}]",
     'XGBoost': f"{loss_xgb.mean():.4f} [{interval(boot['ll_xgb'])[0]:.4f}, {interval(boot['ll_xgb'])[1]:.4f}]",
     'Difference (Naive Bayes minus XGBoost)': f'{ll_difference.mean():+.4f} [{ll_low:+.4f}, {ll_high:+.4f}]',
     'Reading': verdict(ll_low, ll_high, 'XGBoost', 'Naive Bayes')},
    {'Metric': 'ROC AUC', 'Naive Bayes': f"{roc_auc_score(y_test_array, probability_nb):.4f} [{interval(boot['auc_nb'])[0]:.4f}, {interval(boot['auc_nb'])[1]:.4f}]",
     'XGBoost': f"{roc_auc_score(y_test_array, probability_xgb):.4f} [{interval(boot['auc_xgb'])[0]:.4f}, {interval(boot['auc_xgb'])[1]:.4f}]",
     'Difference (Naive Bayes minus XGBoost)': f'{-auc_difference.mean():+.4f} [{-auc_high:+.4f}, {-auc_low:+.4f}]',
     'Reading': verdict(auc_low, auc_high, 'XGBoost', 'Naive Bayes')}])
show(plotly_table(bootstrap_table, 'Paired bootstrap comparison on the test set (95 percent intervals in brackets)', decimals=4,
                  col_widths=[90, 210, 210, 260, 200], height=220))

fig = make_subplots(rows=1, cols=2, subplot_titles=('Test log loss (lower is better)', 'Test ROC AUC (higher is better)'), horizontal_spacing=0.12)
for model_name, key_loss, key_auc, probability in [('Naive Bayes', 'll_nb', 'auc_nb', probability_nb), ('XGBoost', 'll_xgb', 'auc_xgb', probability_xgb)]:
    low, high = interval(boot[key_loss]); point = per_row_log_loss(y_test_array, probability).mean()
    fig.add_trace(go.Scatter(x=[model_name], y=[point], mode='markers+text', name=model_name, legendgroup=model_name,
                             marker=dict(size=18, color=MODEL_COLOURS[model_name]), text=[f'{point:.4f}'], textposition='middle right',
                             error_y=dict(type='data', symmetric=False, array=[high - point], arrayminus=[point - low], color=MODEL_COLOURS[model_name], thickness=3)),
                  row=1, col=1)
    low, high = interval(boot[key_auc]); point = roc_auc_score(y_test_array, probability)
    fig.add_trace(go.Scatter(x=[model_name], y=[point], mode='markers+text', name=model_name, legendgroup=model_name, showlegend=False,
                             marker=dict(size=18, color=MODEL_COLOURS[model_name]), text=[f'{point:.4f}'], textposition='middle right',
                             error_y=dict(type='data', symmetric=False, array=[high - point], arrayminus=[point - low], color=MODEL_COLOURS[model_name], thickness=3)),
                  row=1, col=2)
baseline_loss_test = log_loss(y_test, baseline.predict_proba(X_test[FEATURES])[:, 1])
fig.add_hline(y=baseline_loss_test, line_dash='dot', line_color='#8D99AE', row=1, col=1,
              annotation_text=f'Majority baseline {baseline_loss_test:.4f}', annotation_position='bottom right')
fig.add_hline(y=0.5, line_dash='dot', line_color='#8D99AE', row=1, col=2, annotation_text='Chance 0.500', annotation_position='bottom right')
fig.update_xaxes(range=[-0.6, 1.6])
style_figure(fig, 'Overlapping intervals mean the models cannot be separated', 'Dot is the test result and the bar is the 95 percent bootstrap interval', height=520)
fig.update_layout(legend=dict(orientation='h', y=-0.15))
add_palette_dropdown(fig, trace_ids=[0, 1, 2, 3], groups=[0, 1, 0, 1])
show(fig)

In [31]:
# Model selection rule: lowest validation log loss, but prefer the simpler and more transparent Naive Bayes
# unless XGBoost is clearly better on the test set according to the bootstrap
validation_loss = metrics_validation['Log loss']
test_loss = metrics_test['Log loss']
lower_validation = 'XGBoost' if validation_loss['XGBoost'] < validation_loss['Naive Bayes'] else 'Naive Bayes'
xgb_clearly_better = ll_low > 0
selected_model_name = 'XGBoost' if (lower_validation == 'XGBoost' and xgb_clearly_better) else 'Naive Bayes'
selected_model = models[selected_model_name]
beats_baseline = bool(test_loss[selected_model_name] < test_loss['Majority baseline'])

print(f'Validation log loss: Naive Bayes {validation_loss["Naive Bayes"]:.4f}, XGBoost {validation_loss["XGBoost"]:.4f}, baseline {validation_loss["Majority baseline"]:.4f}')
print(f'Test log loss      : Naive Bayes {test_loss["Naive Bayes"]:.4f}, XGBoost {test_loss["XGBoost"]:.4f}, baseline {test_loss["Majority baseline"]:.4f}')
print(f'Selected model     : {selected_model_name}')
if lower_validation == 'XGBoost' and not xgb_clearly_better:
    print('XGBoost scored slightly lower on validation, but the bootstrap shows no clear difference, so the simpler and more transparent Naive Bayes is preferred.')
elif lower_validation == 'Naive Bayes':
    print('Naive Bayes has the lower validation log loss and is also the simpler and more transparent model.')
else:
    print('XGBoost is clearly better on the test set and is selected.')
print('The selected model beats the majority baseline on test log loss.' if beats_baseline else
      'WARNING: the selected model does not beat the majority baseline, so it adds no reliable value.')

Validation log loss: Naive Bayes 0.6615, XGBoost 0.6619, baseline 0.6702
Test log loss      : Naive Bayes 0.6637, XGBoost 0.6638, baseline 0.6706
Selected model     : Naive Bayes
Naive Bayes has the lower validation log loss and is also the simpler and more transparent model.
The selected model beats the majority baseline on test log loss.


In [32]:
# Which variables drive the predictions? Permutation importance on the test set (increase in log loss when a variable is shuffled)
importance_rows = []
for model_name in ['Naive Bayes', 'XGBoost']:
    result = permutation_importance(models[model_name], X_test[FEATURES], y_test, scoring='neg_log_loss', n_repeats=30,
                                    random_state=SEED, n_jobs=1)
    for feature, mean_value, std_value in zip(FEATURES, result.importances_mean, result.importances_std):
        importance_rows.append({'Model': model_name, 'Variable': feature, 'Increase in log loss': mean_value, 'Std': std_value})
importance = pd.DataFrame(importance_rows)
order = importance[importance['Model'] == 'XGBoost'].sort_values('Increase in log loss', ascending=False)['Variable'].tolist()

fig = go.Figure()
for model_name in ['Naive Bayes', 'XGBoost']:
    part = importance[importance['Model'] == model_name].set_index('Variable').loc[order]
    fig.add_trace(go.Bar(x=[v.replace('_', ' ') for v in order], y=part['Increase in log loss'], name=model_name, marker_color=MODEL_COLOURS[model_name],
                         error_y=dict(type='data', array=part['Std'], color='#495057'), text=[f'{v:.4f}' for v in part['Increase in log loss']],
                         textposition='outside', cliponaxis=False, hovertemplate='%{x}<br>Log loss increase: %{y:.4f}<extra></extra>'))
fig.update_layout(barmode='group', legend=dict(orientation='h', y=-0.2))
style_figure(fig, 'Permutation importance of the governed variables (test set)',
             'Taller bars mean the model relies more on that variable; bars near zero are ignored')
fig.update_yaxes(title='Increase in log loss when shuffled')
add_palette_dropdown(fig, trace_ids=[0, 1])
show(fig)

# XGBoost gain importance on the encoded features
gain = final_xgb.get_booster().get_score(importance_type='gain')
encoded_names = list(preprocessor_xgb.get_feature_names_out())
gain_values = pd.Series({re.sub(r'^(numeric|categorical)__', '', n): gain.get(n, 0.0) for n in encoded_names}).sort_values(ascending=False)
fig = value_bar(gain_values.index.str.replace('_', ' '), gain_values.values, 'XGBoost gain importance of the encoded features',
                'Average improvement in the loss when a split on the feature is used; zero means the feature was never used',
                axis_title='Gain', horizontal=True, height=520, value_format='.3f')
show(fig)

## Fairness audit

Good overall accuracy says nothing about whether the model treats groups fairly, so the predictions are audited for every protected group and for the two named proxy variables (school type and distance from campus).

**How the audit works**
- Each applicant receives an **out-of-fold prediction**: five-fold cross-validation over all 8,000 applicants, so every prediction comes from a model that never saw that applicant. This also gives the small groups enough applicants to measure.
- The shortlist is **capacity matched**: the top share of applicants by predicted probability equal to the historical offer rate is shortlisted. A fixed 0.5 cut-off would shortlist almost everyone and hide any difference.
- Each group is compared with the **largest group** of its characteristic (the reference group).

**Measures**
- **Shortlist rate**: share of the group that the model would shortlist
- **Disparate impact ratio**: group shortlist rate divided by the reference group rate. The four-fifths rule treats a ratio outside **0.80 to 1.25** as a warning sign
- **True positive rate**: among applicants who historically received an offer, the share the model would shortlist (equal opportunity)
- **False positive rate**: among applicants who did not receive an offer, the share the model would shortlist
- Groups with fewer than 200 applicants are marked **Caution** because their rates are statistically noisy

One applicant whose disability status was not recorded is excluded from the audit.

In [33]:
# Out-of-fold predictions for every applicant from tuned versions of both models
cv_audit = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
audit_nb = clone(nb_search.best_estimator_)
audit_xgb = Pipeline([('prep', build_preprocessor(FEATURES)),
                      ('model', XGBClassifier(**{**XGB_BASE, **{**best_params, 'n_estimators': best_round}}))])
oof_probability = {
    'Naive Bayes': cross_val_predict(audit_nb, X_all[FEATURES], y_all, cv=cv_audit, method='predict_proba')[:, 1],
    'XGBoost': cross_val_predict(audit_xgb, X_all[FEATURES], y_all, cv=cv_audit, method='predict_proba')[:, 1],
}

base_rate = y_all.mean()
shortlist = {name: p >= np.quantile(p, 1 - base_rate) for name, p in oof_probability.items()}
print(f'Capacity-matched shortlist size: {base_rate * 100:.1f}% of applicants (the historical offer rate).')
for name in oof_probability:
    print(f'  {name}: out-of-fold log loss {log_loss(y_all, oof_probability[name]):.4f}, ROC AUC {roc_auc_score(y_all, oof_probability[name]):.4f}')


class FairnessAuditor:
    """Group-level fairness measures for a set of predictions."""

    def __init__(self, data, y_true, probability, shortlisted, attributes, min_group_size=200):
        self.frame = data[list(attributes)].copy()
        self.frame['actual'] = np.asarray(y_true)
        self.frame['probability'] = np.asarray(probability)
        self.frame['shortlisted'] = np.asarray(shortlisted).astype(int)
        self.attributes = attributes
        self.min_group_size = min_group_size

    def group_table(self):
        rows = []
        for attribute, label in self.attributes.items():
            part_table = []
            for level, part in self.frame.groupby(attribute):
                positives, negatives = part['actual'] == 1, part['actual'] == 0
                part_table.append({'Characteristic': label, 'Group': str(level), 'Applicants': len(part),
                                   'Historical offer rate': part['actual'].mean(), 'Mean predicted probability': part['probability'].mean(),
                                   'Shortlist rate': part['shortlisted'].mean(),
                                   'True positive rate': part.loc[positives, 'shortlisted'].mean(),
                                   'False positive rate': part.loc[negatives, 'shortlisted'].mean()})
            table = pd.DataFrame(part_table)
            reference = table.loc[table['Applicants'].idxmax()]
            table['Reference group'] = reference['Group']
            table['Disparate impact ratio'] = table['Shortlist rate'] / reference['Shortlist rate']
            table['Shortlist rate gap (points)'] = (table['Shortlist rate'] - reference['Shortlist rate']) * 100
            table['True positive rate gap (points)'] = (table['True positive rate'] - reference['True positive rate']) * 100
            outside = (table['Disparate impact ratio'] < 0.8) | (table['Disparate impact ratio'] > 1.25)
            table['Status'] = np.where(table['Applicants'] < self.min_group_size, 'Caution',
                                       np.where(outside, 'Flag', 'Pass'))
            rows.append(table)
        return pd.concat(rows, ignore_index=True)

    def attribute_summary(self, table):
        rows = []
        for attribute, label in self.attributes.items():
            part = table[table['Characteristic'] == label]
            contingency = pd.crosstab(self.frame[attribute], self.frame['shortlisted'])
            p_value = chi2_contingency(contingency)[1]
            comparable = part[part['Applicants'] >= self.min_group_size]
            worst = comparable.loc[(comparable['Disparate impact ratio'] - 1).abs().idxmax()]
            rows.append({'Characteristic': label, 'Reference group': part['Reference group'].iloc[0],
                         'Group furthest from parity': worst['Group'], 'Disparate impact ratio': worst['Disparate impact ratio'],
                         'Shortlist rate gap (points)': worst['Shortlist rate gap (points)'], 'Chi-square p-value': p_value,
                         'Status': 'Flag' if (part['Status'] == 'Flag').any() else 'Pass'})
        return pd.DataFrame(rows)


audit_frame = X_all.copy()
audit_frame['first_generation_student'] = audit_frame['first_generation_student'].map({0: 'No', 1: 'Yes'})
audit_frame['distance_band'] = pd.cut(audit_frame['distance_from_campus_km'], bins=[-0.1, 50, 100, 150, np.inf],
                                      labels=['Under 50 km', '50 to 100 km', '100 to 150 km', 'Over 150 km']).astype(str)
keep = audit_frame['disability_status'] != 'Not Recorded'
audit_attributes = {'gender': 'Gender', 'ethnicity': 'Ethnicity', 'disability_status': 'Disability', 'first_generation_student': 'First generation',
                    'school_type': 'School type (proxy risk)', 'distance_band': 'Distance (proxy risk)'}

fairness = {}
for name in ['Naive Bayes', 'XGBoost']:
    auditor = FairnessAuditor(audit_frame[keep], y_all[keep], oof_probability[name][keep.values], shortlist[name][keep.values], audit_attributes)
    group_results = auditor.group_table()
    fairness[name] = {'groups': group_results, 'summary': auditor.attribute_summary(group_results)}

selected_summary = fairness[selected_model_name]['summary']
show(plotly_table(selected_summary, f'Fairness summary by characteristic ({selected_model_name}, capacity-matched shortlist)',
                  status_col='Status', decimals=3, col_widths=[170, 130, 170, 150, 150, 130, 80], height=330))

Capacity-matched shortlist size: 60.6% of applicants (the historical offer rate).
  Naive Bayes: out-of-fold log loss 0.6614, ROC AUC 0.5793
  XGBoost: out-of-fold log loss 0.6608, ROC AUC 0.5806


In [34]:
# Heatmap of the main rates for every group (selected model)
groups = fairness[selected_model_name]['groups']
measure_columns = ['Historical offer rate', 'Mean predicted probability', 'Shortlist rate', 'True positive rate', 'False positive rate']
row_labels = [f"{r['Characteristic'].replace(' (proxy risk)', '')}: {r['Group']} (n={r['Applicants']:,})" for _, r in groups.iterrows()]
fig = go.Figure(go.Heatmap(z=groups[measure_columns].values * 100, x=measure_columns, y=row_labels, colorscale='Viridis',
                           texttemplate='%{z:.1f}%', colorbar=dict(title='Percent'),
                           hovertemplate='%{y}<br>%{x}: %{z:.2f}%<extra></extra>'))
style_figure(fig, f'Group-level rates for the selected model ({selected_model_name})',
             'Similar colours down a column mean the model treats groups alike; groups under 200 applicants are noisy', height=900)
fig.update_yaxes(autorange='reversed')
add_cmap_dropdown(fig, 'colorscale')
show(fig)

# Disparate impact ratio for both models with the four-fifths band shaded
labels = [f"{r['Characteristic'].replace(' (proxy risk)', '')}: {r['Group']}" for _, r in groups.iterrows()]
fig = go.Figure()
for name in ['Naive Bayes', 'XGBoost']:
    table = fairness[name]['groups']
    ratio = table['Disparate impact ratio']
    fig.add_trace(go.Bar(y=labels, x=ratio - 1, base=1, orientation='h', name=name, marker_color=MODEL_COLOURS[name],
                         text=[f'{v:.3f}' for v in ratio], textposition='outside', cliponaxis=False, customdata=ratio,
                         hovertemplate='%{y}<br>Disparate impact ratio: %{customdata:.3f}<extra>' + name + '</extra>'))
fig.add_vrect(x0=0.8, x1=1.25, fillcolor='#2A9D8F', opacity=0.12, line_width=0)
fig.add_vline(x=1.0, line_dash='dash', line_color='#1D3557')
fig.update_layout(barmode='group', legend=dict(orientation='h', y=-0.06))
style_figure(fig, 'Disparate impact ratio by group', 'Bars start at parity (1.0) and show how far each group sits from the reference group; the shaded band is the acceptable 0.80 to 1.25 range', height=1100)
fig.update_xaxes(range=[0.5, 1.5], title='Disparate impact ratio'); fig.update_yaxes(autorange='reversed')
add_palette_dropdown(fig, trace_ids=[0, 1])
show(fig)

flagged = {name: int((fairness[name]['groups']['Status'] == 'Flag').sum()) for name in fairness}
cautions = int((groups['Status'] == 'Caution').sum())
print(f'Groups outside the 0.80 to 1.25 band: Naive Bayes {flagged["Naive Bayes"]}, XGBoost {flagged["XGBoost"]}. '
      f'Groups marked Caution because of small size: {cautions}.')

Groups outside the 0.80 to 1.25 band: Naive Bayes 0, XGBoost 0. Groups marked Caution because of small size: 1.


## Transparency: explaining individual predictions

An applicant who is not shortlisted may ask why, and the university must be able to answer in a way that is understandable and defensible. Both models can produce an **applicant-level explanation**.

- **XGBoost** uses its built-in contribution values (the contribution of each feature to the log-odds)
- **Naive Bayes** uses the log-likelihood ratio of each feature, which adds up exactly to the log-odds

For each explanation the contributions plus the starting point (the average log-odds of an offer) add up to the predicted probability. The cell **checks this sum against the model's own prediction**, so the explanation is guaranteed to be faithful. Positive bars push towards an offer and negative bars push away.

In [35]:
def explain_applicant(student_id, model_name, top=10):
    """Explain one prediction as feature contributions in log-odds, verified against the model output."""
    row = X_all.loc[[student_id], FEATURES]
    pipeline = models[model_name]
    encoded = pipeline.named_steps['prep'].transform(row)
    names = [re.sub(r'^(numeric|categorical)__', '', n) for n in encoded.columns]
    if model_name == 'XGBoost':
        booster = pipeline.named_steps['model'].get_booster()
        contributions = booster.predict(xgb.DMatrix(encoded), pred_contribs=True, iteration_range=(0, best_round))[0]
        values, start = contributions[:-1], contributions[-1]
    else:
        classifier, x = pipeline.named_steps['model'], encoded.values[0]
        variance_1, variance_0 = classifier.var_[1], classifier.var_[0]
        values = -0.5 * np.log(variance_1 / variance_0) - 0.5 * ((x - classifier.theta_[1]) ** 2 / variance_1 - (x - classifier.theta_[0]) ** 2 / variance_0)
        start = np.log(classifier.class_prior_[1] / classifier.class_prior_[0])
    probability = 1 / (1 + np.exp(-(start + values.sum())))
    model_probability = pipeline.predict_proba(row)[0, 1]
    assert abs(probability - model_probability) < 1e-3, 'The explanation does not add up to the model prediction'

    def readable(name):
        if name in row.columns:
            return f'{name.replace("_", " ")} = {row[name].iloc[0]:.1f}'
        return f'{name.replace("_", " ")} (applies)'

    table = pd.DataFrame({'Feature': [readable(n) for n in names], 'Contribution': values})
    table = table[(table['Contribution'].abs() > 1e-9) | table['Feature'].str.contains('=')]
    table = table.reindex(table['Contribution'].abs().sort_values(ascending=False).index).head(top)
    return table, start, model_probability


explained = {}
test_probability = pd.Series(models[selected_model_name].predict_proba(X_test[FEATURES])[:, 1], index=X_test.index)
for title, student in [('highest predicted chance of an offer', test_probability.idxmax()), ('lowest predicted chance of an offer', test_probability.idxmin())]:
    table, start, model_probability = explain_applicant(student, selected_model_name)
    explained[title] = (student, model_probability)
    fig = go.Figure(go.Bar(y=table['Feature'], x=table['Contribution'], orientation='h',
                           marker=dict(color=table['Contribution'], colorscale='RdYlGn', cmid=0, showscale=True, colorbar=dict(title='Log-odds')),
                           text=[f'{v:+.3f}' for v in table['Contribution']], textposition='outside', cliponaxis=False,
                           hovertemplate='%{y}<br>Contribution: %{x:+.3f} log-odds<extra></extra>'))
    style_figure(fig, f'Why applicant {student} received this prediction ({selected_model_name})',
                 f'Test applicant with the {title}: predicted probability {model_probability:.1%}; actual outcome: '
                 f'{"offer" if y_all.loc[student] == 1 else "no offer"}', height=520)
    fig.update_xaxes(title='Contribution to the log-odds of an offer (starting point '
                           f'{start:+.3f})'); fig.update_yaxes(autorange='reversed')
    add_cmap_dropdown(fig, 'marker.colorscale', initial='RdYlGn')
    show(fig)
print('Both explanations add up exactly to the model prediction (verified by an assertion in the function).')

Both explanations add up exactly to the model prediction (verified by an assertion in the function).
